# TPPP prospects — `tppp_prospects_v3.ipynb`

Finds third-party payment / payroll processors (TPPPs) in the payment graph, using the ~100 registered TPPP clients
(`customer_type = 'Third-Party Payment/Payroll Processor'` in `vector_customer`) as examples.

**Three lists, three audiences.** Only the second list is prospects in the sales sense.

| List | Who is on it | Table | Who acts |
|---|---|---|---|
| **Look-alikes** (§2–§5) | **PNC TM clients** that move money like registered TPPPs but are not registered | `pkg_cust_metrics_monthly_v3` | TPPP programme / KYC (classification), TM sales (products) |
| **Prospects** (§6–§8) | **counterparties**: outside companies our clients pay or are paid by, split into *new to bank* and *existing or possible PNC client* | `pkg_cpty_metrics_monthly` | TM sales, with warm introductions |
| **Registered TPPPs' business at other banks** (§9) | outside accounts carrying a registered TPPP's name | both | relationship managers |

**How a processor is recognised.** Behaviour only: many payers *and* many payees, regular relationships, money in ≈
money out, counterparties at many banks, low concentration. NAICS codes and name words are **evidence columns**, never
used by the scores. §10 adds **evidence from the data** that does not come from the scores: clients switching from a
known processor to the candidate, employers that stop paying staff directly when they start with it, money in that
leads money out by a day or two, and clients shared with known processors. Each check is first tested on the known
examples.

**Scoring.** Registered TPPPs (look-alikes) or known processors + registered TPPPs' outside accounts (prospects) against
everyone else. Three methods — gradient boosting, logistic regression on percentile ranks, and distance to the nearest
example — each scored **out of fold**, so no row is scored by a model that saw it. Each list ranks by the method with
the best held-out result.

**Status of a hit:** a pattern worth a look, not a finding. Prospecting use needs the permissible-use review that is
already a governance precondition; look-alikes that are processors in fact are also a KYC matter.

**Outputs** (`bdahd01p_dlcdi1_cdi_tm`, external parquet; review and export CSVs in `OUT_DIR`):

| Table | Grain |
|---|---|
| `pkg_tppp_cust_scores` | screened TM organisation (look-alikes): window-average features, scores, flags, evidence, reasons |
| `pkg_tppp_cpty_scores` | screened outside company (prospects): the same, plus relationship to PNC |
| `pkg_tppp_cpty_intros` | top prospects × their largest TM clients |

Runtime: §2, §6 and §10 write their extracts once and reuse them (`REBUILD_EXTRACTS = True` to refresh).

**v2 → v3**

| # | Change | Reason |
|---|---|---|
| 1 | names: *look-alikes* (PNC clients), *prospects* (counterparties), *registered TPPPs' business at other banks* | "prospects" means counterparties only |
| 2 | prospects split into `new to bank`, `existing PNC client` (same name as a PNC organisation) and `possible PNC client` (same distinctive first word) | a company can bank with PNC and elsewhere (Stripe is in both lists); the exact-name flag missed variants like INTUIT PAYROLL S |
| 3 | §9b month by month for the largest registered TPPPs | July and August 2026 carry a doubling of total dollars in the source; check before quoting §9 |
| 4 | §10 evidence from data, validated on the examples, with `evidence_n` and `data_confirmed` | confirm candidates without web access |
| 5 | §11 de-identified export (row id + name, no PNC data) and an internal key file | external enrichment, after sign-off |

**v1 → v2** (after the first real run):

| # | Change | Reason |
|---|---|---|
| 1 | each lens ranks by the method with the best held-out median rank (`FINAL_METHOD = "auto"`) | the blend lost to logit in look-alikes (median 3.2% vs 1.5% ranked above) and to gbm in prospects (19.6% vs 6.5%); the nearest-example method dragged it down |
| 2 | behaviour threshold at the median held-out TPPP (`BEHAVIOUR_TPPP_SHARE = 0.5`) | 0.8 passed 6.6% of the screened population (4,224 names), too many to review |
| 3 | look-alikes list leaves out sectors where no registered TPPP sits (wholesale, retail, manufacturing, food service, utilities, construction, education …), flagged `sector_unlikely` | distributors and fuel wholesalers have many payers, many payees and balanced flow, so they look like processors |
| 4 | 518210 removed from the processor-NAICS seed | it brought in data centres and bitcoin miners |
| 5 | prospects keeps entities whose PNC clients are mostly organisations on at least one side (`CPTY_MIN_ORG_SHARE`), more exclusions (card issuers, auto lenders, insurers, sales tax), flag for generic settlement descriptors | consumer billers (Citi cards, Honda / Hyundai finance, State Farm, sales tax) reached the top 25; names like BNKCD SETTLE or MERCH SVC are processors hidden behind a descriptor |
| 6 | no dollar signs inside table cells; outputs marked to be skipped by the math renderer | JupyterLab rendered text between two dollar signs as math (garbled reasons and introductions) |

## §0 Setup

In [ ]:
# §0a  Session and compat shims
import os, re, json, time, math, hashlib, difflib, warnings, datetime as dt, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
with warnings.catch_warnings():                 # hasattr on removed aliases warns in NumPy 2
    warnings.simplefilter("ignore")
    for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                            ("uint0", np.uintp), ("bool", np.bool_), ("object", object)]:   # np.object: non-Arrow toPandas
        if not hasattr(np, _alias):
            setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
import matplotlib
matplotlib.use("Agg")                           # figures go to disk, not inline
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.utils import AnalysisException
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:                              # sklearn < 1.0
    from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
    from sklearn.ensemble import HistGradientBoostingClassifier

spark = (SparkSession.builder
         .appName("pkg_tppp_prospects")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.memoryOverhead", "3g")
         .config("spark.driver.memory", "8g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "256")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__} | sklearn {sklearn.__version__}")

In [ ]:
# §0b  Config
DB             = "bdahd01p_dlcdi1_cdi_tm"
CUSTOMER_TABLE = "dsihd01p_dsi.vector_customer"
TPPP_TYPE      = "Third-Party Payment/Payroll Processor"
T_CUST_DIM     = f"{DB}.pkg_cust_dim"
T_CUST_MET     = f"{DB}.pkg_cust_metrics_monthly_v3"
T_CUST_DICT    = f"{DB}.pkg_cust_metrics_dictionary_v3"
T_PAIR         = f"{DB}.pkg_cust_pair_month_v3"
T_CPTY_MET     = f"{DB}.pkg_cpty_metrics_monthly"
T_CPTY_DICT    = f"{DB}.pkg_cpty_metrics_dictionary"
T_ENT_DIM      = f"{DB}.pkg_cpty_entity_dim"
T_EDGE_D       = f"{DB}.pkg_edge_daily"
T_MAP          = f"{DB}.pkg_cpty_entity_map_v2"

WINDOW_START, WINDOW_END = "2026-01", "2026-08"   # same window as tppp_scatter_explorer_v2 §4
MIN_ACTIVE_MONTHS = 4          # processors are steady: active in ≥ this many window months
MIN_MATCH_RATE    = 0.5        # fail if fewer registered TPPP power ids than this reach pkg_cust_dim (join-type trap)

# Look-alikes screen (monthly means over active months)
CUST_PARTY_TYPES = ["O"]
CUST_MIN_AMT     = 50_000      # $ received AND $ paid per month
CUST_MIN_CPTY    = 10          # counterparties per month in either direction

# Prospects screen
CPTY_CLASSES  = ["business", "unknown"]
CPTY_MIN_CUST = 20             # PNC customers per month, either direction
CPTY_MIN_ORG_SHARE = 0.5       # $ share with PNC organisation customers, on at least one side (TPPP prospects are B2B)
# Sectors where a TPPP is implausible; a sector is left off the look-alikes list only if no registered TPPP sits in it
UNLIKELY_NAICS2 = ["11", "21", "22", "23", "31", "32", "33", "42", "44", "45", "61", "71", "72"]

# Scoring
SCORE_SET           = "shape"  # "shape": no $ amounts in the models, so lists are not driven by size; "full": all
METHODS             = ["gbm", "logit", "knn"]
N_FOLDS, N_REPEATS  = 5, 2
SEED                = 7
GBM_ITER            = 200
MAX_UNLABELED_TRAIN = 150_000  # unlabeled rows per training fold (cap for speed; scoring always covers everyone)
KNN_MIN_SEP         = 0.05     # features with |AUC − 0.5| below this get no weight in the nearest-example distance
REASON_MIN_SEP      = 0.10     # features eligible as reason codes
BEHAVIOUR_TPPP_SHARE = 0.5     # "behaviour" flag = score at least as high as this share of held-out registered TPPPs
FINAL_METHOD        = "auto"   # "auto" = method with the lowest held-out median rank in each lens; or gbm / logit / knn / blend

# Lists
TOP_N_CUST, TOP_N_CPTY, INTRO_TOP, SHOW_N = 200, 200, 5, 25

# Evidence and exclusions: labels on the lists, never used by the models. Names are matched after
# upper-casing and replacing non-alphanumerics with spaces (BILL.COM → BILL COM).
PROC_NAICS_SEED     = ["522320", "541214", "522390", "561440"]
PROC_NAICS_MIN_TPPP = 3        # NAICS6 codes held by ≥ this many registered TPPPs join the processor-NAICS set
PROC_NAME_RE = (r"\b(?:PAY|PAYS|PAYMENT|PAYMENTS|PAYROLL|PROCESSING|PROCESSOR|PROCESSORS|MERCHANT|MERCHANTS|SETTLEMENT|"
                r"CLEARING|FBO|ESCROW|DISBURSEMENT|DISBURSEMENTS|REMITTANCE|BILLING|COLLECTION|COLLECTIONS)\b")
BANK_NAME_RE = r"\b(?:BANK|BANCORP|BANCSHARES|BANCORPORATION|CREDIT UNION|SAVINGS|FSB)\b"
PNC_NAME_RE  = r"\bPNC\b"
EXCL_NAICS_PREFIX = ["521", "5221", "92"]          # central bank, depository institutions, public administration
KNOWN_PROC_RE = (r"\b(?:ADP|PAYCHEX|PAYCOM|PAYLOCITY|DAYFORCE|CERIDIAN|GUSTO|PAYCOR|ISOLVED|TRINET|JUSTWORKS|INSPERITY|"
                 r"RIPPLING|STRIPE|PAYPAL|BRAINTREE|WORLDPAY|FISERV|FIRST DATA|GLOBAL PAYMENTS|HEARTLAND PAYMENT|TSYS|"
                 r"ELAVON|PAYMENTECH|AUTHORIZE NET|SHIFT4|NUVEI|ADYEN|PAYSAFE|REPAY|DWOLLA|BILL COM|AVIDXCHANGE|"
                 r"CLICKPAY|APPFOLIO|YARDI|PAYLEASE|ZEGO|REALPAGE)\b")    # seed list of well-known processors; edit freely
CPTY_EXCL_RE = (r"\b(?:VISA|MASTERCARD|AMERICAN EXPRESS|AMEX|DISCOVER|IRS|TREASURY|TREAS|SSA|STATE OF|DEPT OF|"
                r"DEPARTMENT OF|COUNTY|CITY OF|COMMONWEALTH|INSURANCE|ASSURANCE|MUTUAL|FANNIE MAE|FREDDIE MAC|"
                r"FEDERAL HOME LOAN|CITICARD|CITI CARDS?|COMENITY|SYNCHRONY|CAPITAL ONE|CHASE CARD|CARD SERVICES|"
                r"HONDA|HYUNDAI|TOYOTA|NISSAN|KIA MOTORS|FORD CREDIT|GM FINANCIAL|STATE FARM|GEICO|PROGRESSIVE|"
                r"ALLSTATE|NATIONWIDE|LIBERTY MUTUAL|USAA)\b"
                r"|SALESTAX|SALES TAX|USETAX|USE TAX|TAXPYMT|DEPT REV")   # networks, government, issuers, lenders, insurers
GENERIC_TOKENS = {"BNKCD", "BANKCARD", "BANKCD", "BNKCRD", "MERCH", "MERCHANT", "MERCHANTS", "MRCH", "SVC", "SVCS",
                  "SERVICE", "SERVICES", "SETTLE", "SETTLEMENT", "STLMT", "DEP", "DEPOSIT", "EMPLOYEE", "BENEFIT",
                  "BENEFITS", "PAYMENT", "PAYMENTS", "PAY", "CARD", "CARDS", "DEBIT", "CREDIT", "ACH", "TRANSFER",
                  "FEE", "FEES", "DISC", "DISCOUNT", "BATCH", "PAYROLL", "REFUND", "PROCESSING", "DEPOSITS", "CHGBK"}

# Relationship of a prospect to PNC (§8)
LOOSE_MAX_FREQ  = 5            # a distinctive first word shared by ≤ this many PNC organisations = similar-name match
FIRST_WORD_STOP = ["THE", "AMERICAN", "AMERICA", "FIRST", "NATIONAL", "UNITED", "GLOBAL", "GENERAL", "INTERNATIONAL",
                   "CITY", "COUNTY", "STATE", "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "PACIFIC", "ATLANTIC",
                   "PREMIER", "ADVANCED", "ALLIED", "STANDARD", "SUPERIOR", "QUALITY", "NEW", "US", "USA", "ALL",
                   "PAY", "PAYROLL", "PAYMENT", "PAYMENTS", "MERCHANT", "SERVICES", "SERVICE", "GROUP", "HOLDINGS"]
NINE_MONTHLY_N  = 8            # §9b: registered TPPPs shown month by month

# Evidence from the data (§10): labels only, never used by the scores; each check is tested on the examples first
EVID_LOOKBACK       = 6        # months before the window used as history (a "new" relationship is absent in them)
SWITCH_MIN_MONTHS   = 3        # a processor relationship lasted ≥ this many months before it stopped
SWITCH_AMT_RATIO    = 3.0      # the new relationship's monthly $ is within 1/3× – 3× of the stopped one
SWITCH_MIN_CLIENTS  = 2        # check passes with ≥ this many switching clients
PERSON_MIN          = 10_000   # client paid ≥ this per month to individuals before starting with the candidate …
PERSON_DROP         = 0.5      # … and at most this share of it in the months after
PAYROLL_MIN_CLIENTS = 2
LAGS                = [1, 2, 3]  # business days by which money in may lead money out
LAG_MIN, LAG_EDGE   = 0.30, 0.10  # lead passes: corr(in today, out L days later) ≥ LAG_MIN and ≥ reverse + LAG_EDGE
LAG_MIN_DAYS        = 20       # weekdays with money in and with money out
OVERLAP_MIN_CLIENTS = 20       # clients-in-common check needs ≥ this many TM clients …
OVERLAP_LIFT        = 1.5      # … that use another known processor ≥ this many times as often as TM clients overall
EVIDENCE_MIN        = 2        # data_confirmed = at least this many checks pass
ENRICH_TOP_N        = 200      # §11 export size per list

OUT_DIR          = "../metrics/tppp/v3"
WRITE_TABLES     = True
REBUILD_EXTRACTS = False
VERSION          = "tppp_v3"
os.makedirs(OUT_DIR, exist_ok=True)
assert SCORE_SET in ("shape", "full") and set(METHODS) <= {"gbm", "logit", "knn"} and METHODS
print(f"window {WINDOW_START} → {WINDOW_END} | score set {SCORE_SET} | methods {METHODS} | outputs {OUT_DIR}")

In [ ]:
# §0c  Helpers: months, display, ids and names, feature resolution, scoring
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def month_range(a, b):
    out, m = [], a
    while m <= b:
        out.append(m)
        m = month_add(m, 1)
    return out

WINDOW = month_range(WINDOW_START, WINDOW_END)

def table_exists(t):
    try:
        spark.table(t).columns
        return True
    except AnalysisException:
        return False

def partitions(table):
    out = set()
    for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
        for kv in r[0].split("/"):
            if kv.startswith("month="):
                out.add(kv.split("=", 1)[1])
    return out

def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")

_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fs(p):
    path = _jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(_hconf), path
def exists(p):
    fs, path = _fs(p)
    return fs.exists(path)
def touch(p):
    fs, path = _fs(p)
    fs.create(path, True).close()

LOC = db_location(DB)
P_TMP = f"{LOC}/_tmp_pkg_tppp"

# ---- display: one output per cell ----
def _fmt(v, name):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "✓" if v else ""
    if isinstance(v, str):
        return v
    if not isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    v = float(v)
    if name.startswith("%") or name.endswith("%") or name.startswith("median %"):
        return f"{v:.1f}"
    if "$" in name:
        return f"{v:,.0f}" if abs(v) >= 10 else f"{v:,.1f}"
    if v.is_integer():
        return f"{v:,.0f}"
    return f"{v:,.2f}" if abs(v) >= 1 else f"{v:.3f}"

def fmt_df(df):
    out = df.copy()
    out.columns = [str(c) for c in out.columns]
    for c in out.columns:
        out[c] = [_fmt(v, c) for v in out[c].tolist()]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 8px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt td:nth-child(-n+3){text-align:left}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#6f6e69;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        elif b[1] is not None:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    # tex2jax_ignore / mathjax_ignore: JupyterLab must not read text between two dollar signs as math
    return HTML(_CSS + "<div class='tex2jax_ignore mathjax_ignore' style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>"
                + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

# ---- ids and names ----
def norm_pwr(c):
    # power id as a 10-digit string with leading zeros (matches tppp_scatter_explorer_v2); '123.0' → '0000000123'
    s = F.regexp_replace(F.trim(c.cast("string")), r"\.0+$", "")
    return F.when(F.length(s) > 0, F.lpad(s, 10, "0"))

LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PC",
         "PLLC", "PA", "PLC", "NA", "THE", "DBA"]
def name_norm(c):
    return F.trim(F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), r"[^A-Z0-9]+", " "))

def name_key(c):
    # same rule as pkg_cpty_entity_dim.pnc_match_n: legal words removed, single letters joined (L L C → LLC)
    k = F.regexp_replace(F.upper(c), r"[^A-Z0-9]+", " ")
    k = F.regexp_replace(F.trim(k), r"\b([A-Z]) (?=[A-Z]\b)", "$1")
    k = F.regexp_replace(k, r"\b(" + "|".join(sorted(LEGAL, key=len, reverse=True)) + r")\b", " ")
    k = F.trim(F.regexp_replace(k, r"\s+", " "))
    return F.when(F.length(k) >= 5, k)

# ---- feature specs ----
def feat(name, cands, label, kind="shape", fmt="num", model=True):
    return dict(name=name, cands=cands, label=label, kind=kind, fmt=fmt, model=model)

def both(name, base, label_in, label_out, kind="shape", fmt="num", model=True, upper=False):
    # one feature per direction; candidates cover lower / upper-case column spellings
    out = []
    for d, lab in (("in", label_in), ("out", label_out)):
        cands = [f"{b}_{d}" for b in (base if isinstance(base, list) else [base])]
        out.append(feat(f"{name}_{d}", cands, lab, kind, fmt, model))
    return out

def resolve(spec, columns, dict_df=None):
    low = {c.lower(): c for c in columns}
    rows, got = [], {}
    for f in spec:
        hit = next((low[c.lower()] for c in f["cands"] if c.lower() in low), None)
        if hit:
            got[f["name"]] = hit
        close = "" if hit else ", ".join(difflib.get_close_matches(f["cands"][0].lower(), list(low), n=3, cutoff=0.6))
        definition = ""
        if hit is not None and dict_df is not None:
            m = dict_df[dict_df.column.str.lower() == hit.lower()]
            definition = str(m.definition.iloc[0]) if len(m) else ""
        rows.append({"feature": f["name"], "column": hit or "MISSING", "in model": f["model"], "kind": f["kind"],
                     "label": f["label"], "closest columns (if missing)": close, "dictionary definition": definition})
    return got, pd.DataFrame(rows)

def read_dict(t):
    if not table_exists(t):
        return None
    d = spark.table(t).toPandas()
    return d if {"column", "definition"} <= set(d.columns) else None

def fmt_value(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "n/a"
    if kind == "usd":       # no dollar sign: cells with two of them render as math in JupyterLab
        return f"{v / 1e6:,.1f}M" if abs(v) >= 1e6 else f"{v / 1e3:,.0f}K" if abs(v) >= 1e3 else f"{v:,.0f}"
    if kind == "pct":
        return f"{100 * v:.0f}%"
    if kind == "ratio":
        return f"{v:.2f}"
    if kind == "int":
        return f"{v:,.0f}"
    return f"{v:,.1f}"

# ---- scoring (shared by look-alikes and prospects) ----
def separation(df, feats, y):
    # single-feature AUC of examples vs the rest, on rows where the feature exists
    rows = []
    for f in feats:
        v = pd.to_numeric(df[f], errors="coerce")
        ok = v.notna().to_numpy()
        auc = np.nan
        if ok[y].sum() >= 3 and ok[~y].sum() >= 3 and v[ok].nunique() > 1:
            auc = roc_auc_score(y[ok], v[ok])
        rows.append({"feature": f, "AUC": auc, "median examples": v[y].median(), "median others": v[~y].median(),
                     "% examples with value": 100 * ok[y].mean(), "% others with value": 100 * ok[~y].mean()})
    s = pd.DataFrame(rows)
    s["separation"] = (s.AUC - 0.5).abs()
    return s.sort_values("separation", ascending=False).reset_index(drop=True)

def rank_matrix(df, feats):
    # percentile rank within the screened population; missing → 0.5 (neutral)
    return df[feats].apply(pd.to_numeric, errors="coerce").rank(pct=True).fillna(0.5).to_numpy(float)

def _fit_predict(method, Xtr, ytr, wtr, Xte, seed):
    if method == "gbm":
        m = HistGradientBoostingClassifier(max_iter=GBM_ITER, learning_rate=0.05, max_leaf_nodes=15,
                                           min_samples_leaf=20, l2_regularization=1.0, random_state=seed)
    else:
        m = LogisticRegression(C=0.5, max_iter=1000)
    m.fit(Xtr, ytr, sample_weight=wtr)
    return m.predict_proba(Xte)[:, 1]

def cv_pct(method, X, y, train_ok, seed0=SEED):
    # Out-of-fold percentile (within the held-out fold), averaged over repeats. Rows with train_ok = False are
    # scored but never trained on (e.g. family members of registered TPPPs, whose status is unknown).
    n, rng = len(y), np.random.default_rng(seed0)
    acc = np.zeros(n)
    for r in range(N_REPEATS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed0 + r)
        pct = np.full(n, np.nan)
        for tr, te in skf.split(np.zeros(n), y):
            tr = tr[train_ok[tr]]
            pos, neg = tr[y[tr]], tr[~y[tr]]
            if len(neg) > MAX_UNLABELED_TRAIN:
                neg = rng.choice(neg, MAX_UNLABELED_TRAIN, replace=False)
            tr = np.concatenate([pos, neg])
            w = np.where(y[tr], len(neg) / max(len(pos), 1), 1.0)       # balance the classes
            s = _fit_predict(method, X[tr], y[tr], w, X[te], seed0 + r)
            pct[te] = pd.Series(s).rank(pct=True).to_numpy()
        acc += pct
    return acc / N_REPEATS

def knn_pct(R, y, w, chunk=50_000):
    # Weighted distance (percentile space) to the nearest example; an example's own row is skipped (leave-one-out).
    pos = np.where(y)[0]
    sw = np.sqrt(w / w.sum())
    P = R[pos] * sw
    pn = (P ** 2).sum(1)
    best, dmin = np.zeros(len(y), int), np.zeros(len(y))
    for a in range(0, len(y), chunk):
        X = R[a:a + chunk] * sw
        d2 = (X ** 2).sum(1)[:, None] + pn[None, :] - 2 * X @ P.T
        rows = np.where((pos >= a) & (pos < a + chunk))[0]
        d2[pos[rows] - a, rows] = np.inf
        best[a:a + chunk] = d2.argmin(1)
        dmin[a:a + chunk] = np.sqrt(np.maximum(d2.min(1), 0))
    return pd.Series(-dmin).rank(pct=True).to_numpy(), pos[best], dmin

def score_population(df, feats, y, train_ok, sep, methods=METHODS):
    # returns {method: percentile}, nearest-example row index, nearest distance
    out, near, dist = {}, None, None
    R = rank_matrix(df, feats)
    X = df[feats].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    for m in methods:
        if m == "gbm":
            out[m] = cv_pct("gbm", X, y, train_ok)
        elif m == "logit":
            out[m] = cv_pct("logit", R, y, train_ok)
    # the nearest example is always computed: it is the explanation shown on every list
    s = sep.set_index("feature").reindex(feats)
    w = np.clip(s.separation.fillna(0).to_numpy() - KNN_MIN_SEP, 0, None)
    w = w if w.sum() > 0 else np.ones(len(feats))
    kp, near, dist = knn_pct(R, y, w)
    if "knn" in methods:
        out["knn"] = kp
    out["blend"] = np.nanmean(np.vstack([out[m] for m in methods]), axis=0)
    return out, near, dist

KS = [0.001, 0.005, 0.01, 0.05]
def recall_table(pcts, y, groups=None):
    # held-out rank of the examples: share of the screened population ranked above each one
    rows, N = [], len(y)
    for name, p in pcts.items():
        for g, mask in (groups or {"all examples": y}).items():
            q = 1 - p[mask]
            if len(q) == 0:
                continue
            r = {"method": name, "examples": g, "n": int(mask.sum()), "median % ranked above": 100 * np.median(q)}
            for k in KS:
                r[f"found in top {100 * k:g}% ({int(round(k * N)):,} rows)"] = int((q <= k).sum())
            rows.append(r)
    return pd.DataFrame(rows)

def pick_method(rec, pcts, group="all examples"):
    # FINAL_METHOD, or the scoring method with the lowest held-out median rank (ties → first in METHODS + blend)
    if FINAL_METHOD != "auto":
        assert FINAL_METHOD in pcts, f"FINAL_METHOD {FINAL_METHOD} not scored; have {list(pcts)}"
        return FINAL_METHOD
    r = rec[rec.method.isin(list(pcts)) & (rec.examples == group)]
    return r.sort_values("median % ranked above").method.iloc[0]

def recall_figure(pcts, y, path, title):
    xs = np.logspace(-4, 0, 200)
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for name, p in pcts.items():
        q = np.sort(1 - p[y])
        ax.plot(xs, [(q <= x).mean() for x in xs], label=name, lw=2 if name == "blend" else 1.2)
    ax.plot(xs, xs, ls=":", c="grey", label="random")
    ax.set_xscale("log")
    ax.set_xlabel("share of screened population reviewed (top-ranked first)")
    ax.set_ylabel("share of held-out examples found")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

def reasons(df, feats, sep, spec_by_name, top=3):
    # Reason codes: the features on which a row is most example-like, as oriented percentiles among the screened
    s = sep.set_index("feature").reindex(feats)
    use = [f for f in feats if s.loc[f, "separation"] >= REASON_MIN_SEP]
    if not use:
        return pd.Series([""] * len(df), index=df.index)
    V = df[use].apply(pd.to_numeric, errors="coerce")
    R = V.rank(pct=True)
    O = np.where((s.loc[use, "AUC"] > 0.5).to_numpy()[None, :], R.to_numpy(), 1 - R.to_numpy())
    O = np.nan_to_num(O, nan=0.0)
    idx = np.argsort(-O, axis=1)[:, :top]
    out = []
    for i in range(len(df)):
        parts = []
        for j in idx[i]:
            f = use[j]
            if O[i, j] < 0.75:
                continue
            sp = spec_by_name[f]
            top_ = 100 * (1 - O[i, j])
            parts.append(f"{sp['label'].replace('$ ', '')} {fmt_value(V.iat[i, j], sp['fmt'])} "
                         f"(top {'<0.1' if top_ < 0.1 else f'{top_:.1f}'}%)")
        out.append("; ".join(parts))
    return pd.Series(out, index=df.index)

def to_sdf(pdf):
    # explicit schema; NaN → NULL in text columns
    typ = {"object": "string", "bool": "boolean", "int64": "bigint", "int32": "int", "float64": "double",
           "Int64": "bigint", "boolean": "boolean"}
    pdf = pdf.copy()
    for c in pdf.columns:
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
    schema = ", ".join(f"`{c}` {typ.get(str(pdf[c].dtype), 'string')}" for c in pdf.columns)
    return spark.createDataFrame(pdf, schema)

def write_table(pdf, name):
    path = f"{LOC}/{name}"
    to_sdf(pdf).repartition(4).write.mode("overwrite").parquet(path)
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{name}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(path).schema.fields)
    spark.sql(f"CREATE EXTERNAL TABLE {DB}.{name} ({cols}) STORED AS PARQUET LOCATION '{path}'")
    return path

def extract(path, sig, build):
    # write once, read back; rebuilt when the signature changes or REBUILD_EXTRACTS is set
    mark = f"{path}/_pkg_sig_{sig}"
    rebuilt = REBUILD_EXTRACTS or not (exists(path + "/_SUCCESS") and exists(mark))
    if rebuilt:
        build().write.mode("overwrite").parquet(path)
        touch(mark)
    return spark.read.parquet(path), rebuilt

print(f"window months: {len(WINDOW)} | scratch {P_TMP}")

## §1 Registered TPPPs

Latest `load_date` snapshot of `vector_customer`; `cust_pwr_id` normalised to a 10-digit string with leading zeros on
both sides of the join to `pkg_cust_dim`. Family members (same `master_pwr_id`) are kept apart: they are neither
examples nor prospects. The NAICS codes the registered TPPPs carry extend the processor-NAICS evidence set.

In [ ]:
# §1  Registered TPPP power ids → mdm_ids
tppp_ids = (
    spark.sql(f'''
        select cust_pwr_id
        from {CUSTOMER_TABLE}
        where customer_type = '{TPPP_TYPE}'
          and load_date = (select max(load_date) from {CUSTOMER_TABLE})
    ''')
    .select(norm_pwr(F.col("cust_pwr_id")).alias("customer_pwr_id"))
    .where(F.col("customer_pwr_id").isNotNull())
    .distinct()
    .withColumn("is_tppp", F.lit(True))
    .cache()
)
n_tppp_ids = tppp_ids.count()

CDIM = (spark.table(T_CUST_DIM)
        .select("mdm_id", norm_pwr(F.col("customer_pwr_id")).alias("customer_pwr_id"),
                norm_pwr(F.col("master_pwr_id")).alias("master_pwr_id"), "customer_name", "party_type",
                "naics_cd", "naics_desc", "naics2", "naics_status",
                F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"),
                F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)).alias("pwr_shared")))
tppp_m = tppp_ids.join(CDIM, "customer_pwr_id", "left").cache()
TPPP = tppp_m.where(F.col("mdm_id").isNotNull()).toPandas()
n_matched = TPPP.customer_pwr_id.nunique()
assert n_tppp_ids > 0, f"no rows with customer_type = '{TPPP_TYPE}' in the latest snapshot of {CUSTOMER_TABLE}"
assert n_matched >= MIN_MATCH_RATE * n_tppp_ids, (
    f"only {n_matched} of {n_tppp_ids} TPPP power ids found in {T_CUST_DIM}: check the id format on both sides")
unmatched = tppp_m.where(F.col("mdm_id").isNull()).select("customer_pwr_id").toPandas()

fam_ids = set(TPPP.master_pwr_id.dropna())
FAMILY = (CDIM.where(F.col("master_pwr_id").isin(list(fam_ids)) if fam_ids else F.lit(False))
              .select("mdm_id").toPandas())
FAMILY_IDS = set(FAMILY.mdm_id) - set(TPPP.mdm_id)
TPPP_IDS = set(TPPP.mdm_id)

TPPP["naics6"] = np.where(TPPP.naics_status == "valid", TPPP.naics_cd.astype(str).str[:6], None)
nc = (TPPP.dropna(subset=["naics6"]).groupby(["naics6"]).agg(n=("mdm_id", "nunique"), desc=("naics_desc", "first"))
          .reset_index().sort_values("n", ascending=False))
PROC_NAICS = sorted(set(PROC_NAICS_SEED) | set(nc.loc[nc.n >= PROC_NAICS_MIN_TPPP, "naics6"]))
counts = pd.DataFrame([
    {"item": "TPPP power ids in latest snapshot", "n": n_tppp_ids},
    {"item": "found in pkg_cust_dim", "n": n_matched},
    {"item": "mdm_ids (a power id can hold several)", "n": len(TPPP_IDS)},
    {"item": "power ids shared by 2+ mdm_ids (pwr_shared)", "n": int(TPPP.loc[TPPP.pwr_shared, "customer_pwr_id"].nunique())},
    {"item": "TM customers", "n": int(TPPP.is_tm.sum())},
    {"item": "family members kept apart (same master_pwr_id)", "n": len(FAMILY_IDS)},
])
pt = TPPP.party_type.fillna("(null)").value_counts().rename_axis("party_type").reset_index(name="mdm_ids")
show(f"§1 Registered TPPPs: {n_matched} of {n_tppp_ids} power ids matched",
     ("Counts", counts),
     ("Party type", pt),
     ("NAICS of registered TPPPs (valid codes)", nc.rename(columns={"naics6": "naics", "n": "TPPPs"}).head(15)),
     note(f"Processor-NAICS evidence set: {', '.join(PROC_NAICS)} (seed + codes held by ≥ {PROC_NAICS_MIN_TPPP} TPPPs)."),
     ("Unmatched power ids (first 20)", unmatched.head(20)) if len(unmatched) else None)

## §2 Look-alikes (PNC clients) — features and extract

Features are resolved against the actual columns of `pkg_cust_metrics_monthly_v3` (case-insensitive, with fallbacks)
and shown with their dictionary definitions. A missing feature is dropped and named, with the closest columns, so a
one-line fix in `CUST_SPEC` is enough. Direction is the customer's view: `_in` = received.

In [ ]:
# §2a  Customer feature spec → table columns
CUST_SPEC = [
    feat("amt_in", ["amt_all_in"], "$ received /mo", "size", "usd"),
    feat("amt_out", ["amt_all_out"], "$ paid /mo", "size", "usd"),
    feat("passthrough", ["passthrough_ratio"], "in/out balance", fmt="ratio"),
    *both("ncpty", "ncpty_all", "payers /mo", "payees /mo", fmt="int"),
    *both("top1", "top1_share", "largest payer share", "largest payee share", fmt="pct"),
    *both("person", "ext_person_share", "external $ in from persons", "external $ out to persons", fmt="pct"),
    *both("ext", "ext_share", "$ in from outside PNC", "$ out to outside PNC", fmt="pct"),
    *both("naics_ent", "naics2_entropy", "industry spread of PNC payers", "industry spread of PNC payees"),
    *both("days", "active_days", "days with money in /mo", "days with money out /mo"),
    *both("ticket", "avg_ticket", "average payment in", "average payment out", fmt="usd"),
    *both("ach", ["share_ACH", "share_ach"], "ACH share in", "ACH share out", fmt="pct"),
    *both("wire", ["share_WIRE", "share_wire"], "wire share in", "wire share out", fmt="pct"),
    *both("rails", "n_rails", "rails in", "rails out"),
    *both("nfi", "n_fi", "banks of external payers", "banks of external payees", fmt="int"),
    *both("retention", "retention", "payer retention", "payee retention", fmt="pct"),
    *both("rec", "rec_standing", "recurring payers", "recurring payees", fmt="int"),
    *both("new_amt", "new_amt_share", "$ in from new payers", "$ out to new payees", fmt="pct"),
    *both("selfpay", "selfpay_share", "$ in from own accounts elsewhere", "$ out to own accounts elsewhere", fmt="pct"),
    # not in the models: used in §9
    *both("amt_int", "amt_int", "$ in from PNC customers", "$ out to PNC customers", "size", "usd", model=False),
    *both("amt_ext", "amt_ext", "$ in from outside PNC", "$ out to outside PNC", "size", "usd", model=False),
]
CUST_CORE = ["amt_in", "amt_out", "ncpty_in", "ncpty_out"]
CUST_BY_NAME = {f["name"]: f for f in CUST_SPEC}
CUST_DICT = read_dict(T_CUST_DICT)
CUST_COL, cres = resolve(CUST_SPEC, spark.table(T_CUST_MET).columns, CUST_DICT)
miss_core = [f for f in CUST_CORE if f not in CUST_COL]
assert not miss_core, f"core customer features missing: {miss_core} — fix CUST_SPEC (see the table)"
CUST_FEATS = [f["name"] for f in CUST_SPEC if f["model"] and f["name"] in CUST_COL]
CUST_MODEL_FEATS = [f for f in CUST_FEATS if SCORE_SET == "full" or CUST_BY_NAME[f]["kind"] != "size"]
have = partitions(T_CUST_MET)
miss_m = [m for m in WINDOW if m not in have]
assert not miss_m, f"{T_CUST_MET}: window months missing {miss_m}"
n_miss = int((cres.column == "MISSING").sum())
show(f"§2a Customer features: {len(CUST_COL)} of {len(CUST_SPEC)} resolved, {len(CUST_MODEL_FEATS)} used by the models",
     ("Resolution", cres.sort_values(["column"], key=lambda s: s != "MISSING", ascending=True) if n_miss else cres),
     note(f"Score set '{SCORE_SET}': " + ("$ amounts are left out of the models (size alone does not make a processor)."
                                          if SCORE_SET == "shape" else "all features in the models.")),
     note("No dictionary table found: definitions empty.") if CUST_DICT is None else None)

In [ ]:
# §2b  Window averages per TM organisation (written once), screen, funnel
cols_used = sorted(set(CUST_COL.values()))
SIG_C = hashlib.md5(json.dumps({"v": VERSION, "w": WINDOW, "c": cols_used, "p": CUST_PARTY_TYPES}).encode()).hexdigest()[:10]

def build_cust_agg():
    pop = CDIM.where(F.col("is_tm") & F.col("party_type").isin(CUST_PARTY_TYPES)).select("mdm_id")
    m = (spark.table(T_CUST_MET).where(F.col("month").isin(WINDOW))
              .select("mdm_id", "month", *[F.col(c).cast("double").alias(c) for c in cols_used])
              .join(pop, "mdm_id", "left_semi"))
    act = (F.coalesce(F.col(CUST_COL["amt_in"]), F.lit(0.0)) + F.coalesce(F.col(CUST_COL["amt_out"]), F.lit(0.0))) > 0
    aggs = [F.sum(act.cast("int")).alias("months_active")]
    aggs += [F.avg(F.when(act, F.col(CUST_COL[f]))).alias(f) for f in CUST_COL]
    return m.groupBy("mdm_id").agg(*aggs).where(F.col("months_active") > 0)

cagg, rebuilt_c = extract(f"{P_TMP}/cust_agg", SIG_C, build_cust_agg)
lab = spark.createDataFrame(pd.DataFrame({"mdm_id": sorted(TPPP_IDS | FAMILY_IDS)}).assign(
          is_tppp=lambda d: d.mdm_id.isin(TPPP_IDS), tppp_family=lambda d: ~d.mdm_id.isin(TPPP_IDS)))
cagg_l = cagg.join(F.broadcast(lab), "mdm_id", "left")
cagg_l = (cagg_l.withColumn("is_tppp", F.coalesce("is_tppp", F.lit(False)))
                .withColumn("tppp_family", F.coalesce("tppp_family", F.lit(False))))
s1 = F.col("months_active") >= MIN_ACTIVE_MONTHS
s2 = s1 & (F.col("amt_in") >= CUST_MIN_AMT) & (F.col("amt_out") >= CUST_MIN_AMT)
s3 = s2 & ((F.col("ncpty_in") >= CUST_MIN_CPTY) | (F.col("ncpty_out") >= CUST_MIN_CPTY))
fun = cagg_l.agg(*[F.sum(c.cast("int")).alias(f"n{i}") for i, c in enumerate([F.lit(True), s1, s2, s3])],
                 *[F.sum((c & F.col("is_tppp")).cast("int")).alias(f"t{i}") for i, c in enumerate([F.lit(True), s1, s2, s3])]
                 ).first()
tppp_tm_org = int((TPPP.is_tm & TPPP.party_type.isin(CUST_PARTY_TYPES)).sum())
funnel = pd.DataFrame([
    {"step": f"registered TPPPs that are TM, party {CUST_PARTY_TYPES}", "TM organisations": None, "TPPPs": tppp_tm_org},
    {"step": f"active in {WINDOW_START} → {WINDOW_END}", "TM organisations": fun["n0"], "TPPPs": fun["t0"]},
    {"step": f"active ≥ {MIN_ACTIVE_MONTHS} months", "TM organisations": fun["n1"], "TPPPs": fun["t1"]},
    {"step": f"≥ {CUST_MIN_AMT:,} received and paid per month", "TM organisations": fun["n2"], "TPPPs": fun["t2"]},
    {"step": f"≥ {CUST_MIN_CPTY} payers or payees per month (screened)", "TM organisations": fun["n3"], "TPPPs": fun["t3"]},
])
funnel["% of TPPPs kept"] = 100 * funnel.TPPPs / max(tppp_tm_org, 1)

C = (cagg_l.where(s3 | F.col("is_tppp") | F.col("tppp_family"))
           .withColumn("screened", s3)
           .join(CDIM.select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "naics_cd", "naics_desc",
                             "naics_status", name_norm(F.col("customer_name")).alias("name_norm"),
                             name_key(F.col("customer_name")).alias("name_key")), "mdm_id", "left")
           .toPandas())
C["naics6"] = np.where(C.naics_status == "valid", C.naics_cd.astype(str).str[:6], None)
nn = C.name_norm.fillna("")
C["naics_proc"] = C.naics6.isin(PROC_NAICS)
C["name_proc"] = nn.str.contains(PROC_NAME_RE, regex=True)
C["pnc_own"] = nn.str.contains(PNC_NAME_RE, regex=True)
C["bank_like"] = nn.str.contains(BANK_NAME_RE, regex=True) | C.naics6.fillna("").str.startswith(tuple(EXCL_NAICS_PREFIX[:2]))
C["gov"] = C.naics6.fillna("").str.startswith(tuple(p for p in EXCL_NAICS_PREFIX if p.startswith("92")))
C["excluded"] = C.pnc_own | C.bank_like | C.gov
C["naics2v"] = C.naics6.fillna("").str[:2]
TPPP_SECTORS = set(C.loc[C.is_tppp & (C.naics2v != ""), "naics2v"])
EXCL_SECTORS = sorted(set(UNLIKELY_NAICS2) - TPPP_SECTORS)
C["sector_unlikely"] = C.naics2v.isin(EXCL_SECTORS)
CS = C[C.screened].reset_index(drop=True)          # the scoring population
lost = C[C.is_tppp & ~C.screened]
lost_tbl = lost.assign(**{"$ in /mo": lost.amt_in, "$ out /mo": lost.amt_out})[
    ["customer_name", "naics_cd", "months_active", "$ in /mo", "$ out /mo", "ncpty_in", "ncpty_out"]].head(15)
show(f"§2b Look-alikes population: {len(CS):,} screened TM organisations, {int(CS.is_tppp.sum())} registered TPPPs among them"
     f" ({'built' if rebuilt_c else 'reused'} extract)",
     ("Funnel", funnel),
     note("The screen should keep most TPPPs; if it drops many, loosen CUST_MIN_AMT / CUST_MIN_CPTY."),
     ("Registered TPPPs dropped by the screen (first 15)", lost_tbl) if len(lost) else None)

## §3 What a registered TPPP looks like

Each feature on its own: how well it separates registered TPPPs from the rest of the screened population
(AUC 0.5 = no signal, 1.0 = every TPPP above every other customer, 0.0 = every TPPP below). Medians are of monthly
averages over the window. This is also where a feature's direction for the reason codes comes from.

In [ ]:
# §3  Single-feature separation, registered TPPPs vs screened others
yC = CS.is_tppp.to_numpy(bool)
assert yC.sum() >= 10, f"only {yC.sum()} registered TPPPs in the screened population: loosen the screen"
SEP_C = separation(CS, CUST_FEATS, yC)
tbl = SEP_C.assign(label=SEP_C.feature.map(lambda f: CUST_BY_NAME[f]["label"]),
                   **{"in model": SEP_C.feature.isin(CUST_MODEL_FEATS)})
fig, ax = plt.subplots(figsize=(6.5, 0.22 * len(tbl) + 1))
ax.barh(tbl.label[::-1], tbl.AUC[::-1] - 0.5, left=0.5, color=np.where(tbl.AUC[::-1] > 0.5, "#2a6f97", "#c44e52"))
ax.axvline(0.5, c="grey", lw=0.8)
ax.set_xlim(0, 1)
ax.set_xlabel("AUC: registered TPPPs vs other screened TM organisations")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/cust_feature_auc.png", dpi=130)
plt.close(fig)
show(f"§3 Registered TPPPs ({int(yC.sum())}) vs other screened TM organisations ({int((~yC).sum()):,})",
     ("Features by separation", tbl[["feature", "label", "AUC", "median examples", "median others",
                                     "% examples with value", "in model"]].rename(
         columns={"median examples": "median TPPP", "median others": "median other"})),
     note(f"Figure: {OUT_DIR}/cust_feature_auc.png"))

## §4 Look-alikes — scoring and validation

Every screened customer gets an out-of-fold percentile from each method; registered TPPPs are scored by models that
never saw them, so their ranks are an honest test. Family members of registered TPPPs are scored but never trained on.
The `gbm` ablation on the other feature set shows how much size alone contributes.

Baselines: random review finds k% of TPPPs in the top k%; the best single feature is the bar any model must clear.

In [ ]:
# §4  Out-of-fold scores, held-out recall of registered TPPPs, ablation
t0 = time.time()
handle = display(render(note("scoring…")), display_id=True)
train_ok = ~CS.tppp_family.to_numpy(bool)
PCT_C, near_c, dist_c = score_population(CS, CUST_MODEL_FEATS, yC, train_ok, SEP_C)
best = SEP_C[SEP_C.feature.isin(CUST_FEATS)].iloc[0]
rb = CS[best.feature].rank(pct=True).fillna(0.5).to_numpy()
evals = dict(PCT_C)
evals[f"single feature: {best.feature}"] = rb if best.AUC > 0.5 else 1 - rb
other = [f for f in CUST_FEATS if CUST_BY_NAME[f]["kind"] != "size"] if SCORE_SET == "full" else CUST_FEATS
handle.update(render(note("ablation…")))
abl = cv_pct("gbm", CS[other].apply(pd.to_numeric, errors="coerce").to_numpy(float), yC, train_ok)
evals[f"gbm on '{'shape' if SCORE_SET == 'full' else 'full'}' set (ablation)"] = abl
REC_C = recall_table(evals, yC)
recall_figure({k: v for k, v in evals.items() if "ablation" not in k}, yC, f"{OUT_DIR}/cust_recall.png",
              "Look-alikes: held-out registered TPPPs found vs share reviewed")
for m, p in PCT_C.items():
    CS[f"pct_{m}"] = p
FINAL_C = pick_method(REC_C, PCT_C)
CS["score"] = pd.Series(PCT_C[FINAL_C]).rank(pct=True).to_numpy()   # percentile of the chosen method
CS["screened_rank"] = CS.score.rank(ascending=False, method="min").astype(int)
CS["nearest_tppp"] = CS.customer_name.to_numpy()[near_c]
CS["nearest_tppp_dist"] = dist_c
CS["reasons"] = reasons(CS, CUST_MODEL_FEATS, SEP_C, CUST_BY_NAME)
THR_C = float(np.quantile(CS.score[yC], 1 - BEHAVIOUR_TPPP_SHARE))
CS["behaviour"] = CS.score >= THR_C
CS["declared"] = CS.naics_proc | CS.name_proc
handle.update(render(
    f"§4 Look-alikes scores ({(time.time() - t0) / 60:.1f} min): {len(CS):,} customers, {int(yC.sum())} held-out TPPPs",
    ("Held-out recall (higher counts and lower median = better)", REC_C),
    note("Ranks are percentiles within each held-out fold, averaged over repeats; 'median % ranked above' is the "
         "typical share of the screened population scoring above a hidden TPPP."),
    note(f"Lists rank by '{FINAL_C}' (FINAL_METHOD = {FINAL_METHOD!r})."),
    note(f"Behaviour threshold: score ≥ {100 * THR_C:.1f} — as high as {100 * BEHAVIOUR_TPPP_SHARE:g}% of held-out "
         f"registered TPPPs; {100 * (CS.score >= THR_C).mean():.1f}% of the screened population passes."),
    note(f"Figure: {OUT_DIR}/cust_recall.png")))

## §5 Look-alikes — review list

Registered TPPPs, their family members, PNC's own entities, banks and public administration are left off the list
(they stay in the scores table). Triage combines behaviour (a score at least as high as `BEHAVIOUR_TPPP_SHARE` of
the held-out registered TPPPs) with the evidence columns:

| triage | meaning |
|---|---|
| `behaviour + declared` | behaves like a TPPP and its NAICS or name says processor: strongest |
| `behaviour only` | behaves like a TPPP, nothing declared: review |
| `declared only` | NAICS or name says processor, behaviour does not: a KYC classification check rather than a prospect |

In [ ]:
# §5  Look-alikes prospect list and the declared-only check list
CS["triage"] = np.select([CS.behaviour & CS.declared, CS.behaviour & CS.sector_unlikely, CS.behaviour, CS.declared],
                         ["behaviour + declared", "behaviour, sector unlikely", "behaviour only", "declared only"],
                         default="")
cand_all = CS[~CS.is_tppp & ~CS.tppp_family & ~CS.excluded]
cand = cand_all[~cand_all.sector_unlikely | cand_all.declared].sort_values("score", ascending=False)
LIST_C = cand.head(TOP_N_CUST).copy()
LIST_C.insert(0, "rank", np.arange(1, len(LIST_C) + 1))

def cust_view(d):
    return pd.DataFrame({
        "rank": d["rank"] if "rank" in d else np.nan, "customer": d.customer_name, "power id": d.customer_pwr_id,
        "naics": d.naics_cd.fillna("").astype(str) + " " + d.naics_desc.fillna("").astype(str).str[:30],
        "triage": d.triage, "evidence": d["evidence"] if "evidence" in d else "", "screened rank": d.screened_rank, "$K in /mo": d.amt_in / 1e3, "$K out /mo": d.amt_out / 1e3,
        "payers": d.ncpty_in.round(), "payees": d.ncpty_out.round(), "in/out": d.passthrough,
        "nearest TPPP": d.nearest_tppp, "reasons": d.reasons})

trc = cand_all.triage.replace("", "(below the behaviour threshold, nothing declared)").value_counts().rename_axis("triage").reset_index(name="customers")
sec = (CS.assign(s=CS.naics2v.replace("", "(none)")).groupby("s")
         .agg(screened=("mdm_id", "size"), TPPPs=("is_tppp", "sum"),
              behaviour=("behaviour", lambda b: int((b & ~CS.loc[b.index, "is_tppp"]).sum())))
         .reset_index().rename(columns={"s": "naics2"}).sort_values("screened", ascending=False))
sec["off the list"] = sec.naics2.isin(EXCL_SECTORS)
decl = cand[cand.triage == "declared only"].sort_values("amt_in", ascending=False)
show(f"§5 Look-alikes: top {len(LIST_C)} of {len(cand):,} candidate TM organisations",
     ("Triage of all candidates", trc),
     ("Sectors (NAICS2): screened, registered TPPPs, behaviour hits; sectors with no TPPP are off the list",
      sec.head(25)),
     (f"Top {SHOW_N} prospects (full list in §10 outputs)", cust_view(LIST_C.head(SHOW_N))),
     (f"Declared processor, not registered, below the behaviour threshold — KYC check (top {SHOW_N} by $ received)",
      cust_view(decl.head(SHOW_N)).drop(columns=["rank"])) if len(decl) else None,
     note(f"Left off the list: {int(CS.excluded.sum()):,} PNC-own / bank / public-administration rows, "
          f"{int(CS.tppp_family.sum()):,} TPPP family members, "
          f"{int((cand_all.sector_unlikely & ~cand_all.declared).sum()):,} in sectors {', '.join(EXCL_SECTORS)} "
          f"(kept in pkg_tppp_cust_scores with sector_unlikely = true)."))

## §6 Prospects (counterparties) — outside companies

`pkg_cpty_metrics_monthly` rows that are eligible, of class `business` or `unknown`, averaged over the window.
Direction is the counterparty's view: `_in` = the entity receives from PNC customers.

Examples for prospects: well-known processors (`KNOWN_PROC_RE`, a seed list to edit) and the **outside accounts of
registered TPPPs** (entity name key equal to a registered TPPP's name key). Entities matching `CPTY_EXCL_RE` (card
networks, government, insurers, GSEs) are never examples and never prospects.

In [ ]:
# §6a  Counterparty feature spec → table columns
CPTY_SPEC = [
    *both("amt", "amt", "$ from PNC clients /mo", "$ to PNC clients /mo", "size", "usd"),
    *both("ncust", "n_cust", "PNC clients paying it", "PNC clients it pays", fmt="int"),
    *both("ncust_tm", "n_cust_tm", "TM clients paying it", "TM clients it pays", fmt="int"),
    *both("org", "org_share", "$ in from organisations", "$ out to organisations", fmt="pct"),
    *both("tm", "tm_share", "$ in from TM clients", "$ out to TM clients", fmt="pct"),
    *both("top1", "top1_share", "largest payer share", "largest payee share", fmt="pct"),
    *both("naics_ent", "naics2_entropy", "industry spread of payers", "industry spread of payees"),
    *both("dep", "dep_wavg", "share of payers' outside flow", "share of payees' outside flow", fmt="pct"),
    feat("two_way", ["two_way_ratio"], "two-way balance", fmt="ratio"),
    *both("days", "active_days", "days with money in /mo", "days with money out /mo"),
    *both("ticket", "avg_ticket", "average payment in", "average payment out", fmt="usd"),
    *both("retention", "retention", "payer retention", "payee retention", fmt="pct"),
    *both("rec", "rec_standing", "recurring payers", "recurring payees", fmt="int"),
    *both("new_amt", "new_amt_share", "$ in from new payers", "$ out to new payees", fmt="pct"),
    *both("ach", "share_ach", "ACH share in", "ACH share out", fmt="pct"),
    *both("wire", "share_wire", "wire share in", "wire share out", fmt="pct"),
    *both("rails", "n_rails", "rails in", "rails out"),
    feat("nfi", ["n_fi"], "banks holding its accounts", fmt="int"),
    feat("topfi", ["top_fi_share"], "share at its main bank", fmt="pct"),
    feat("nids", ["n_ids_active"], "account ids active", fmt="int"),
    feat("ncust_all", ["n_cust_all"], "PNC clients either way", fmt="int", model=False),
]
CPTY_DERIVED = [feat("ach_orig", [], "months as ACH originator", fmt="pct"),
                feat("role_flip", [], "org share in − out", fmt="ratio")]
CPTY_BY_NAME = {f["name"]: f for f in CPTY_SPEC + CPTY_DERIVED}
CPTY_DICT = read_dict(T_CPTY_DICT)
CPTY_COL, pres = resolve(CPTY_SPEC, spark.table(T_CPTY_MET).columns, CPTY_DICT)
miss_core = [f for f in ["amt_in", "amt_out", "ncust_in", "ncust_out", "ncust_all"] if f not in CPTY_COL]
assert not miss_core, f"core counterparty features missing: {miss_core}"
need = {"ent_id", "ent_name", "name_class", "cpty_type", "fi_dom", "eligible", "amt_all", "month"}
assert need <= set(spark.table(T_CPTY_MET).columns), f"{T_CPTY_MET} lacks {need - set(spark.table(T_CPTY_MET).columns)}"
miss_m = [m for m in WINDOW if m not in partitions(T_CPTY_MET)]
assert not miss_m, f"{T_CPTY_MET}: window months missing {miss_m}"
CPTY_FEATS = ([f["name"] for f in CPTY_SPEC if f["model"] and f["name"] in CPTY_COL]
              + (["ach_orig"]) + (["role_flip"] if {"org_in", "org_out"} <= set(CPTY_COL) else []))
CPTY_MODEL_FEATS = [f for f in CPTY_FEATS if SCORE_SET == "full" or CPTY_BY_NAME[f]["kind"] != "size"]
show(f"§6a Counterparty features: {len(CPTY_COL)} of {len(CPTY_SPEC)} resolved, {len(CPTY_MODEL_FEATS)} used by the models",
     ("Resolution", pres),
     note("Derived: ach_orig = share of active months with cpty_type NON_PNC_ACH_ORIGINATOR; "
          "role_flip = org share in − org share out (payroll: in from employers, out to people)."))

In [ ]:
# §6b  Window averages per entity (written once), labels, screen, funnel
cols_p = sorted(set(CPTY_COL.values()))
SIG_P = hashlib.md5(json.dumps({"v": VERSION, "w": WINDOW, "c": cols_p, "k": CPTY_CLASSES}).encode()).hexdigest()[:10]

def build_cpty_agg():
    m = (spark.table(T_CPTY_MET).where(F.col("month").isin(WINDOW))
              .where(F.col("eligible") & F.col("name_class").isin(CPTY_CLASSES) & (F.col("amt_all") > 0))
              .select("ent_id", "month", "ent_name", "name_class", "cpty_type", "fi_dom",
                      F.col("amt_all").cast("double").alias("_amt"),
                      *[F.col(c).cast("double").alias(c) for c in cols_p]))
    aggs = [F.count("*").alias("months_active"),
            F.max(F.struct("_amt", "ent_name")).getField("ent_name").alias("ent_name"),
            F.max(F.struct("month", "name_class")).getField("name_class").alias("name_class"),
            F.max(F.struct("month", "fi_dom")).getField("fi_dom").alias("fi_dom"),
            F.avg((F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR").cast("double")).alias("ach_orig")]
    aggs += [F.avg(F.col(CPTY_COL[f])).alias(f) for f in CPTY_COL]
    return (m.groupBy("ent_id").agg(*aggs)
             .withColumn("name_norm", name_norm(F.col("ent_name")))
             .withColumn("name_key", name_key(F.col("ent_name"))))

pagg, rebuilt_p = extract(f"{P_TMP}/cpty_agg", SIG_P, build_cpty_agg)
tk = (spark.createDataFrame(TPPP[["mdm_id", "customer_name"]].dropna())
           .select("mdm_id", name_key(F.col("customer_name")).alias("name_key")).where(F.col("name_key").isNotNull()))
tkeys = tk.select("name_key").distinct().withColumn("tppp_ext", F.lit(True))
ent = spark.table(T_ENT_DIM).select("ent_id", "pnc_match_n", "pnc_tm_match_n", F.col("n_ids").alias("n_ids_lifetime"))
pl = (pagg.join(F.broadcast(tkeys), "name_key", "left").join(ent, "ent_id", "left")
          .withColumn("tppp_ext", F.coalesce("tppp_ext", F.lit(False)))
          .withColumn("known_proc", F.col("name_norm").rlike(KNOWN_PROC_RE))
          .withColumn("cpty_excl", F.col("name_norm").rlike(CPTY_EXCL_RE)))
q1 = F.col("months_active") >= MIN_ACTIVE_MONTHS
q2a = q1 & (F.col("ncust_all") >= CPTY_MIN_CUST)
org_max = F.greatest(F.coalesce(F.col("org_in"), F.lit(0.0)), F.coalesce(F.col("org_out"), F.lit(0.0)))
q2 = q2a & (org_max >= CPTY_MIN_ORG_SHARE) & ~F.col("cpty_excl")
steps = [("eligible business / unknown entities active in the window", F.lit(True)),
         (f"active ≥ {MIN_ACTIVE_MONTHS} months", q1), (f"≥ {CPTY_MIN_CUST} PNC clients per month", q2a),
         (f"≥ {100 * CPTY_MIN_ORG_SHARE:g}% of $ with organisations on one side, not excluded (screened)", q2)]
fr = pl.agg(*[F.sum(c.cast("int")).alias(f"n{i}") for i, (_, c) in enumerate(steps)],
            *[F.sum((c & F.col("known_proc") & ~F.col("cpty_excl")).cast("int")).alias(f"k{i}") for i, (_, c) in enumerate(steps)],
            *[F.sum((c & F.col("tppp_ext") & ~F.col("cpty_excl")).cast("int")).alias(f"t{i}") for i, (_, c) in enumerate(steps)]).first()
pfun = pd.DataFrame([{"step": s, "entities": fr[f"n{i}"], "known processors": fr[f"k{i}"],
                      "registered TPPPs' outside accounts": fr[f"t{i}"]} for i, (s, _) in enumerate(steps)])
P = pl.where(q2).toPandas()
if {"org_in", "org_out"} <= set(P.columns):
    P["role_flip"] = P.org_in - P.org_out
P["example"] = (P.known_proc | P.tppp_ext) & ~P.cpty_excl
P["pnc_name_match"] = P.pnc_match_n.fillna(0) > 0
P["generic_name"] = P.name_norm.fillna("").map(lambda n: bool(n.split()) and set(n.split()) <= GENERIC_TOKENS)
show(f"§6b Prospects population: {len(P):,} screened entities, {int(P.example.sum())} examples "
     f"({'built' if rebuilt_p else 'reused'} extract)",
     ("Funnel", pfun),
     ("Examples: known processors (first 15 by PNC clients)",
      P[P.known_proc & ~P.cpty_excl].sort_values("ncust_all", ascending=False)
        .assign(**{"$K in /mo": lambda d: d.amt_in / 1e3, "$K out /mo": lambda d: d.amt_out / 1e3})
        [["ent_name", "fi_dom", "ncust_in", "ncust_out", "$K in /mo", "$K out /mo"]].head(15)),
     note(f"Registered TPPPs with an outside account matched by name key: "
          f"{tk.join(tkeys, 'name_key').join(pagg.select('name_key'), 'name_key').select('mdm_id').distinct().count()} "
          f"of {len(TPPP_IDS)} (all entities, before the screen; see §9)."))

## §7 Prospects — scoring and validation

Same machinery as §4. Recall is shown separately for the two kinds of example: the known-processor seed list is
the closer stand-in for a prospect (a processor banking elsewhere).

In [ ]:
# §7  Out-of-fold scores for external entities
t0 = time.time()
handle = display(render(note("scoring…")), display_id=True)
yP = P.example.to_numpy(bool)
assert yP.sum() >= 10, f"only {yP.sum()} examples among screened entities: extend KNOWN_PROC_RE or loosen the screen"
SEP_P = separation(P, CPTY_FEATS, yP)
PCT_P, near_p, dist_p = score_population(P, CPTY_MODEL_FEATS, yP, np.ones(len(P), bool), SEP_P)
bestp = SEP_P.iloc[0]
rbp = P[bestp.feature].rank(pct=True).fillna(0.5).to_numpy()
evp = dict(PCT_P)
evp[f"single feature: {bestp.feature}"] = rbp if bestp.AUC > 0.5 else 1 - rbp
groups = {"known processors": yP & P.known_proc.to_numpy(bool),
          "TPPP outside accounts": yP & P.tppp_ext.to_numpy(bool) & ~P.known_proc.to_numpy(bool)}
REC_P = recall_table(evp, yP, groups)
recall_figure(evp, yP, f"{OUT_DIR}/cpty_recall.png", "Prospects: held-out processor examples found vs share reviewed")
for m, p in PCT_P.items():
    P[f"pct_{m}"] = p
FINAL_P = pick_method(REC_P, PCT_P, group="known processors")
P["score"] = pd.Series(PCT_P[FINAL_P]).rank(pct=True).to_numpy()
P["screened_rank"] = P.score.rank(ascending=False, method="min").astype(int)
P["nearest_example"] = P.ent_name.to_numpy()[near_p]
P["nearest_example_dist"] = dist_p
P["reasons"] = reasons(P, CPTY_MODEL_FEATS, SEP_P, CPTY_BY_NAME)
sep_tbl = SEP_P.assign(label=SEP_P.feature.map(lambda f: CPTY_BY_NAME[f]["label"])).head(15)
handle.update(render(
    f"§7 Prospects scores ({(time.time() - t0) / 60:.1f} min): {len(P):,} entities, {int(yP.sum())} held-out examples",
    ("Held-out recall by kind of example", REC_P),
    note(f"Lists rank by '{FINAL_P}' (best held-out median rank on known processors; FINAL_METHOD = {FINAL_METHOD!r})."),
    ("Most separating features (examples vs other screened entities)",
     sep_tbl[["feature", "label", "AUC", "median examples", "median others"]]),
    note(f"Figure: {OUT_DIR}/cpty_recall.png")))

## §8 Prospects — list, relationship to PNC, warm introductions

Top outside companies that are not registered TPPPs' own outside accounts and not excluded. Known processors stay on
the list, flagged. Each prospect gets a **relationship to PNC**, from its name against every PNC organisation:

| relationship | rule | conversation |
|---|---|---|
| `existing PNC client` | same cleaned name as a PNC organisation (legal words removed) | wallet share: bring the accounts held elsewhere |
| `possible PNC client` | same distinctive first word as ≤ `LOOSE_MAX_FREQ` PNC organisations (INTUIT PAYROLL S ↔ INTUIT INC) | check the matched names first |
| `new to bank` | neither | new-to-bank prospect |

Name matching misses clients known under another name, so `new to bank` is "no PNC client found by name". Warm
introductions are the prospect's largest TM clients over the window, from `pkg_cust_pair_month_v3` (`paid` = the TM
client pays the prospect).

In [ ]:
# §8  Prospects list, relationship to PNC, largest TM clients of each prospect
# ---- relationship to PNC by name: exact cleaned name, else a rare distinctive first word ----
stop_sql = ", ".join("'" + w + "'" for w in FIRST_WORD_STOP)
def first_word(col):
    return F.expr(f"element_at(filter(split({col}, ' '), t -> length(t) >= 3 AND NOT array_contains(array({stop_sql}), t)), 1)")
orgs = (CDIM.where(F.col("party_type") == "O")
            .select("mdm_id", "customer_name", "is_tm", name_key(F.col("customer_name")).alias("k"))
            .where(F.col("k").isNotNull()).withColumn("fw", first_word("k")))
fw_freq = orgs.groupBy("fw").agg(F.count("*").alias("fw_n")).where(F.col("fw_n") <= LOOSE_MAX_FREQ)
pk = (spark.createDataFrame(P[["ent_id", "name_key"]].dropna()).withColumnRenamed("name_key", "k")
          .withColumn("fw", first_word("k")))
ex = (pk.join(orgs.select("k", "customer_name", "is_tm"), "k")
        .groupBy("ent_id").agg(F.count("*").alias("n_exact"), F.sum(F.col("is_tm").cast("int")).alias("n_exact_tm"),
                               F.expr("max_by(customer_name, cast(is_tm as int))").alias("exact_name")))
lo = (pk.join(ex.select("ent_id"), "ent_id", "left_anti").join(fw_freq, "fw")
        .join(orgs.select("fw", "customer_name", "is_tm"), "fw")
        .groupBy("ent_id").agg(F.count("*").alias("n_loose"),
                               F.concat_ws("; ", F.slice(F.sort_array(F.collect_set("customer_name")), 1, 3)).alias("loose_names")))
REL = ex.join(lo, "ent_id", "full").toPandas()
P = P.drop(columns=[c for c in ["n_exact", "n_exact_tm", "exact_name", "n_loose", "loose_names"] if c in P]) \
     .merge(REL, on="ent_id", how="left")
P["relationship"] = np.select(
    [(P.n_exact.fillna(0) > 0) | P.pnc_name_match, P.n_loose.fillna(0) > 0],
    ["existing PNC client", "possible PNC client"], default="new to bank")
P["pnc_match_names"] = P.exact_name.fillna(P.loose_names)

candp = P[~P.tppp_ext & ~P.cpty_excl].sort_values("score", ascending=False)
LIST_P = candp.head(TOP_N_CPTY).copy()
LIST_P["org_in"] = LIST_P.get("org_in", np.nan)
LIST_P["org_out"] = LIST_P.get("org_out", np.nan)
LIST_P.insert(0, "rank", np.arange(1, len(LIST_P) + 1))

pc = {c.lower(): c for c in spark.table(T_PAIR).columns}
def pcol(*cands):
    hit = next((pc[c] for c in cands if c in pc), None)
    assert hit, f"{T_PAIR}: none of {cands}"
    return hit
P_CUST, P_DIR, P_SCOPE, P_OTHER, P_AMT = (pcol("cust", "mdm_id"), pcol("dir"), pcol("scope"),
                                          pcol("other", "other_id"), pcol("amount", "amt"))
ids = spark.createDataFrame(LIST_P[["ent_id"]])
pr = (spark.table(T_PAIR).where(F.col("month").isin(WINDOW) & (F.col(P_SCOPE) == "ext"))
           .join(F.broadcast(ids), F.col(P_OTHER) == F.col("ent_id"))
           .groupBy("ent_id", F.col(P_CUST).alias("mdm_id"))
           .agg(F.sum(F.when(F.col(P_DIR) == "out", F.col(P_AMT))).cast("double").alias("paid"),
                F.sum(F.when(F.col(P_DIR) == "in", F.col(P_AMT))).cast("double").alias("received"),
                F.countDistinct("month").alias("months"))
           .join(CDIM.where("is_tm").select("mdm_id", "customer_name", "customer_pwr_id", "master_pwr_id",
                                             "naics_cd"), "mdm_id"))
INTRO = pr.toPandas().fillna({"paid": 0.0, "received": 0.0})
INTRO["total"] = INTRO.paid + INTRO.received
ntm = INTRO.groupby("ent_id").agg(n_tm=("mdm_id", "nunique"), n_fam=("master_pwr_id", "nunique"))
INTRO = INTRO.sort_values(["ent_id", "total"], ascending=[True, False])
INTRO["intro_rank"] = INTRO.groupby("ent_id").cumcount() + 1
INTRO = INTRO[INTRO.intro_rank <= INTRO_TOP]
t3 = INTRO[INTRO.intro_rank <= 3]
top3 = pd.Series({e: "; ".join(f"{n} ({t / 1e3:,.0f}K)" for n, t in zip(d.customer_name, d.total))
                  for e, d in t3.groupby("ent_id")}, dtype=object, name="largest TM clients")
LIST_P = LIST_P.join(ntm, on="ent_id").join(top3, on="ent_id")

def cpty_view(d):
    return pd.DataFrame({
        "rank": d["rank"], "entity": d.ent_name, "relationship": d.relationship, "PNC name match": d.pnc_match_names,
        "known processor": d.known_proc, "generic name": d.generic_name, "bank today": d.fi_dom,
        "evidence": d["evidence"] if "evidence" in d else "",
        "banks": d.nfi.round(1) if "nfi" in d else np.nan, "screened rank": d.screened_rank,
        "org % in": 100 * d.org_in, "org % out": 100 * d.org_out,
        "PNC clients in": d.ncust_in.round(), "PNC clients out": d.ncust_out.round(), "TM clients": d.n_tm,
        "$K in /mo": d.amt_in / 1e3, "$K out /mo": d.amt_out / 1e3,
        "nearest example": d.nearest_example, "reasons": d.reasons, "largest TM clients": d["largest TM clients"]})

rel_tbl = (pd.crosstab(LIST_P.relationship, LIST_P.known_proc.map({True: "known processor", False: "other"}),
                       margins=True, margins_name="all").reset_index())
show(f"§8 Prospects: top {len(LIST_P)} of {len(candp):,} outside companies",
     (f"Relationship to PNC, top {len(LIST_P)}", rel_tbl),
     (f"New to bank: top {SHOW_N}", cpty_view(LIST_P[LIST_P.relationship == "new to bank"].head(SHOW_N))),
     (f"Existing or possible PNC client: top {SHOW_N} (check the matched names)",
      cpty_view(LIST_P[LIST_P.relationship != "new to bank"].head(SHOW_N))),
     note(f"Generic settlement descriptors in the top 50: {int(LIST_P.head(50).generic_name.sum())} — the processor "
          "behind them is identified by the sponsor bank and routing / ACH company id, not by the name."),
     note(f"Known processors in the top 50: {int(LIST_P.head(50).known_proc.sum())}. Warm-introduction rows: "
          f"{len(INTRO):,} (up to {INTRO_TOP} TM clients per prospect, window totals)."))

## §9 Registered TPPPs — their business at other banks

For each registered TPPP, the outside accounts that carry its name (same name key) and the money PNC clients send
there, next to what PNC clients pay its PNC accounts. `share elsewhere` = outside ÷ (outside + PNC), both as monthly
averages over the window. A same-name match can be wrong for generic names; the matched entity name is shown.

In [ ]:
# §9  Registered TPPPs: outside accounts by name key
mt = (tk.join(pagg.select("ent_id", "name_key", "ent_name", "fi_dom", "months_active", "amt_in", "amt_out",
                          "ncust_in", "ncust_out"), "name_key").toPandas())
own = C[C.is_tppp][["mdm_id", "customer_name", "customer_pwr_id"] +
                   [c for c in ["amt_int_in", "amt_int_out", "amt_in", "amt_out"] if c in C.columns]]
if len(mt):
    g = (mt.groupby("mdm_id")
           .agg(entities=("ent_id", "nunique"), outside_entity=("ent_name", "first"),
                outside_bank=("fi_dom", lambda s: ", ".join(sorted(set(s.dropna()))[:3])),
                out_in=("amt_in", "sum"), out_out=("amt_out", "sum"), clients_in=("ncust_in", "sum"))
           .reset_index())
    W9 = own.merge(g, on="mdm_id", how="inner")
    int_in = W9.amt_int_in if "amt_int_in" in W9 else np.nan
    W9["share_elsewhere"] = W9.out_in / (W9.out_in + int_in)
    W9 = W9.sort_values("out_in", ascending=False)
    v9 = pd.DataFrame({"TPPP": W9.customer_name, "power id": W9.customer_pwr_id, "outside entity": W9.outside_entity,
                       "entities": W9.entities, "bank(s) outside": W9.outside_bank,
                       "$K/mo PNC clients → its PNC accounts": int_in / 1e3,
                       "$K/mo PNC clients → its outside accounts": W9.out_in / 1e3,
                       "PNC clients paying outside": W9.clients_in, "% elsewhere": 100 * W9.share_elsewhere})
else:
    W9, v9 = pd.DataFrame(), pd.DataFrame()
show(f"§9 Registered TPPPs with outside accounts carrying their name: {len(W9)} of {len(TPPP_IDS)}",
     ("By money PNC clients send to the outside accounts", v9.head(SHOW_N)) if len(v9) else None,
     note("Not matched ≠ no outside accounts: names in the payment feeds often differ from the customer name. "
          "Prefix or alias matching is a later step."))

In [ ]:
# §9b  Registered TPPPs month by month: PNC clients → its PNC accounts vs its accounts elsewhere
if len(W9):
    top9 = W9.head(NINE_MONTHLY_N)[["mdm_id", "customer_name"]]
    k_int = top9.assign(scope="int", other=top9.mdm_id)
    k_ext = mt[mt.mdm_id.isin(top9.mdm_id)][["mdm_id", "ent_id"]].rename(columns={"ent_id": "other"}).assign(scope="ext")
    k9 = spark.createDataFrame(pd.concat([k_int[["mdm_id", "scope", "other"]], k_ext[["mdm_id", "scope", "other"]]]))
    m9 = (spark.table(T_PAIR).where(F.col("month").isin(WINDOW) & (F.col(P_DIR) == "out"))
               .select(F.col(P_SCOPE).alias("scope"), F.col(P_OTHER).alias("other"), "month",
                       F.col(P_AMT).cast("double").alias("amt"))
               .join(F.broadcast(k9), ["scope", "other"])
               .groupBy("mdm_id", "scope", "month").agg(F.sum("amt").alias("amt")).toPandas())
    m9 = m9.merge(top9, on="mdm_id")
    m9["to"] = m9.scope.map({"int": "PNC accounts", "ext": "elsewhere"})
    pv = (m9.pivot_table(index=["customer_name", "to"], columns="month", values="amt", aggfunc="sum")
            .reindex(columns=WINDOW).fillna(0) / 1e6)
    late, early = WINDOW[-2:], WINDOW[:-2]
    pv["last 2 mo ÷ earlier"] = pv[late].mean(axis=1) / pv[early].mean(axis=1).replace(0, np.nan)
    tot = m9.groupby("month").amt.sum().reindex(WINDOW).fillna(0) / 1e6
    V9B = pv.reset_index().rename(columns={"customer_name": "TPPP", "to": "PNC clients pay into"})
    V9B.columns = [f"{c} $M" if c in WINDOW else c for c in V9B.columns]
    show(f"§9b Month by month, {len(top9)} largest registered TPPPs by money elsewhere ($M per month)",
         ("Monthly", V9B),
         note("A 'last 2 mo ÷ earlier' far above 1 means July–August drive the average: quote the January–June "
              "level instead until the source jump is explained."))
else:
    V9B = pd.DataFrame()
    show("§9b No registered TPPP with same-name outside accounts")

## §10 Evidence from the data

Four checks that use the data in a different way from the scores. Each one is computed for every screened look-alike
and prospect, and **first tested on the known examples**: a check is only worth something if registered TPPPs (or
known processors) pass it much more often than everyone else.

| Check | Rule | Why it points to a processor |
|---|---|---|
| `switch` | ≥ `SWITCH_MIN_CLIENTS` TM clients stopped a steady relationship (≥ 3 months) with a known processor or registered TPPP and, within −1 to +2 months, started one with the candidate at a similar monthly amount (1/3× – 3×) | clients replace one payroll or payment provider by another |
| `payroll` | ≥ `PAYROLL_MIN_CLIENTS` TM clients started paying the candidate while their own payments to individuals fell by at least half (from ≥ 10K a month) | the candidate took over paying their staff |
| `lead` | on weekdays, money in today correlates with money out 1–3 days later (≥ `LAG_MIN`), clearly more than the reverse | funding arrives, then pay-outs go out |
| `shared` | its TM clients use another known processor ≥ `OVERLAP_LIFT` times as often as TM clients overall | it serves the same kind of client, in the same ecosystem |

`evidence_n` counts passed checks; `data_confirmed` = at least `EVIDENCE_MIN`. History starts `EVID_LOOKBACK` months
before the window, so "new" means not seen in that history. Direction is each client's: `out` = the client pays.

In [ ]:
# §10a  Monthly relationship checks: switch, payroll, shared clients
EXT_MONTHS = month_range(month_add(WINDOW_START, -EVID_LOOKBACK), WINDOW_END)
miss_m = [m for m in EXT_MONTHS if m not in partitions(T_PAIR)]
assert not miss_m, f"{T_PAIR}: months missing for the evidence history {miss_m}"
MI = {m: i for i, m in enumerate(EXT_MONTHS)}
I_W0, I_END = MI[WINDOW_START], MI[WINDOW_END]
MIDX = F.create_map(*[F.lit(x) for kv in MI.items() for x in kv])

REF = pd.concat([
    TPPP[["mdm_id", "customer_name"]].rename(columns={"mdm_id": "other", "customer_name": "ref_name"}).assign(scope="int"),
    pl.where((F.col("known_proc") | F.col("tppp_ext")) & ~F.col("cpty_excl"))
      .select(F.col("ent_id").alias("other"), F.col("ent_name").alias("ref_name")).toPandas().assign(scope="ext"),
]).drop_duplicates(["scope", "other"])
CAND = pd.concat([CS[["mdm_id"]].rename(columns={"mdm_id": "other"}).assign(scope="int"),
                  P[["ent_id"]].rename(columns={"ent_id": "other"}).assign(scope="ext")]).drop_duplicates()
KEYS = (CAND.assign(is_cand=True).merge(REF.assign(is_ref=True), on=["scope", "other"], how="outer")
            .fillna({"is_cand": False, "is_ref": False}))
KEYS_S = spark.createDataFrame(KEYS[["scope", "other", "is_cand", "is_ref"]].astype({"is_cand": bool, "is_ref": bool}))
TMORG = CDIM.where(F.col("is_tm") & F.col("party_type").isin(CUST_PARTY_TYPES)).select(F.col("mdm_id").alias("cust"))
PAIR = (spark.table(T_PAIR).where(F.col("month").isin(EXT_MONTHS))
             .select(F.col(P_CUST).alias("cust"), F.col(P_DIR).alias("dir"), F.col(P_SCOPE).alias("scope"),
                     F.col(P_OTHER).alias("other"), "month", F.col(P_AMT).cast("double").alias("amt")))
SIG_E = hashlib.md5(json.dumps({"v": VERSION, "m": EXT_MONTHS, "k": len(KEYS),
                                "h": hashlib.md5("".join(sorted(KEYS.scope + KEYS.other)).encode()).hexdigest()}
                               ).encode()).hexdigest()[:10]

def build_rel():
    r = (PAIR.where(F.col("scope").isin("int", "ext")).join(F.broadcast(KEYS_S), ["scope", "other"])
             .join(TMORG, "cust", "left_semi").where(F.col("cust") != F.col("other"))
             .withColumn("mi", MIDX[F.col("month")]))
    return r.groupBy("cust", "dir", "scope", "other").agg(
        F.min("mi").alias("first"), F.max("mi").alias("last"), F.countDistinct("mi").alias("n_months"),
        F.avg("amt").alias("amt"), F.first("is_cand").alias("is_cand"), F.first("is_ref").alias("is_ref"))

t0 = time.time()
REL_E, rebuilt_e = extract(f"{P_TMP}/evid_rel", SIG_E, build_rel)
refn = spark.createDataFrame(REF[["scope", "other", "ref_name"]]).select(
    F.col("scope").alias("ref_scope"), F.col("other").alias("ref_other"), "ref_name")

# ---- switch ----
lost = (REL_E.where(F.col("is_ref") & (F.col("n_months") >= SWITCH_MIN_MONTHS)
                    & F.col("last").between(I_W0 - 1, I_END - 2))
             .select("cust", "dir", F.col("scope").alias("ref_scope"), F.col("other").alias("ref_other"),
                     F.col("last").alias("ref_last"), F.col("amt").alias("ref_amt")))
new = REL_E.where(F.col("is_cand") & (F.col("first") >= I_W0) & (F.col("n_months") >= 2))
sw = (new.join(lost, ["cust", "dir"])
         .where(((F.col("scope") != F.col("ref_scope")) | (F.col("other") != F.col("ref_other")))
                & F.col("first").between(F.col("ref_last") - 1, F.col("ref_last") + 2)
                & (F.col("amt") / F.col("ref_amt")).between(1 / SWITCH_AMT_RATIO, SWITCH_AMT_RATIO))
         .join(refn, ["ref_scope", "ref_other"], "left"))
ev_sw = (sw.groupBy("scope", "other", "ref_name").agg(F.countDistinct("cust").alias("c"))
           .groupBy("scope", "other").agg(F.expr("max_by(ref_name, c)").alias("switch_from"))
           .join(sw.groupBy("scope", "other").agg(F.countDistinct("cust").alias("switch_n")), ["scope", "other"]))

# ---- payroll moved ----
pcols = {c.lower(): c for c in spark.table(T_PAIR).columns}
o_cls, o_pty = pcols.get("o_class"), pcols.get("o_party_type")
HAVE_PERSON = o_cls is not None or o_pty is not None
if HAVE_PERSON:
    is_person = F.lit(False)
    if o_cls:
        is_person = is_person | ((F.col(P_SCOPE) == "ext") & (F.upper(F.col(o_cls)) == "PERSON"))
    if o_pty:
        is_person = is_person | ((F.col(P_SCOPE) == "int") & (F.col(o_pty) == "P"))
    new_out = new.where((F.col("dir") == "out") & (F.col("first") <= I_END - 1)).select("cust", "scope", "other", "first")
    persons = (spark.table(T_PAIR).where(F.col("month").isin(EXT_MONTHS) & (F.col(P_DIR) == "out") & is_person)
                    .select(F.col(P_CUST).alias("cust"), MIDX[F.col("month")].alias("mi"), F.col(P_AMT).cast("double").alias("p"))
                    .join(new_out.select("cust").distinct(), "cust", "left_semi")
                    .groupBy("cust", "mi").agg(F.sum("p").alias("p")))
    n_after = F.least(F.col("first") + 3, F.lit(I_END)) - F.col("first")
    pm = (new_out.join(persons, "cust", "left")
                 .groupBy("scope", "other", "cust", "first")
                 .agg(F.sum(F.when(F.col("mi").between(F.col("first") - 3, F.col("first") - 1), F.col("p"))).alias("b"),
                      F.sum(F.when(F.col("mi").between(F.col("first") + 1, F.col("first") + 3), F.col("p"))).alias("a"))
                 .withColumn("before", F.coalesce("b", F.lit(0.0)) / 3)
                 .withColumn("after", F.coalesce("a", F.lit(0.0)) / n_after)
                 .where((F.col("before") >= PERSON_MIN) & (F.col("after") <= PERSON_DROP * F.col("before"))))
    ev_pm = pm.groupBy("scope", "other").agg(F.countDistinct("cust").alias("payroll_n"))
else:
    ev_pm = spark.createDataFrame([], "scope string, other string, payroll_n long")

# ---- shared clients ----
relw = REL_E.where(F.col("last") >= I_W0)
nref = relw.where("is_ref").groupBy("cust").agg(F.countDistinct("scope", "other").alias("n_ref"))
n_active = cagg.count()
base_share = nref.count() / max(n_active, 1)
cc = (relw.where("is_cand").select("cust", "scope", "other", "is_ref").distinct()
          .join(nref, "cust", "left").fillna(0, ["n_ref"])
          .withColumn("other_ref", (F.col("n_ref") - F.col("is_ref").cast("int")) > 0))
ev_ov = (cc.groupBy("scope", "other").agg(F.count("*").alias("shared_clients"),
                                          F.avg(F.col("other_ref").cast("double")).alias("shared_share"))
           .withColumn("shared_lift", F.col("shared_share") / F.lit(base_share)))

EV_M = (ev_ov.join(ev_sw, ["scope", "other"], "full").join(ev_pm, ["scope", "other"], "full").toPandas())
show(f"§10a Monthly relationship checks ({(time.time() - t0) / 60:.1f} min, {'built' if rebuilt_e else 'reused'} extract)",
     ("Inputs", pd.DataFrame([
         {"item": "history months", "value": f"{EXT_MONTHS[0]} → {EXT_MONTHS[-1]}"},
         {"item": "known processors / registered TPPPs used as reference", "value": f"{len(REF):,}"},
         {"item": "candidates (look-alikes + prospects)", "value": f"{len(CAND):,}"},
         {"item": "TM organisations active in the window that use a reference processor",
          "value": f"{100 * base_share:.1f}%"},
         {"item": "candidates with ≥ 1 switching client / payroll-moved client",
          "value": f"{int((EV_M.switch_n.fillna(0) > 0).sum()):,} / {int((EV_M.payroll_n.fillna(0) > 0).sum()):,}"}])),
     note("payroll check skipped: no o_class / o_party_type column in the pair table") if not HAVE_PERSON else None)

In [ ]:
# §10b  Money in leads money out (weekday series from pkg_edge_daily)
WEEKDAYS = [d for d in pd.date_range(f"{WINDOW_START}-01", pd.Period(WINDOW_END).end_time.date()) if d.weekday() < 5]
N_WD = len(WEEKDAYS)
CAL = spark.createDataFrame(pd.DataFrame({"txn_date": [d.date() for d in WEEKDAYS], "wi": np.arange(N_WD)}))
ca = spark.createDataFrame(CS[["mdm_id"]]).withColumnRenamed("mdm_id", "other")
cb = spark.createDataFrame(P[["ent_id"]]).withColumnRenamed("ent_id", "other")
SIG_D = hashlib.md5(json.dumps({"v": VERSION, "w": WINDOW, "a": len(CS), "b": len(P),
                                "h": hashlib.md5("".join(sorted(CAND.other)).encode()).hexdigest()}).encode()).hexdigest()[:10]

def build_daily():
    d = (spark.table(T_EDGE_D).where(F.col("month").isin(WINDOW) & ~F.coalesce(F.col("is_self"), F.lit(False)))
              .select("txn_date", "src_id", "src_kind", "dst_id", "dst_kind", F.col("amount").cast("double").alias("amt")))
    a = (d.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("other"), F.lit("out").alias("dir"), "txn_date", "amt")
          .unionByName(d.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("other"), F.lit("in").alias("dir"),
                                                                   "txn_date", "amt"))
          .join(F.broadcast(ca), "other").withColumn("scope", F.lit("int")))
    mp = (spark.table(T_MAP).groupBy("cpty_id").agg(F.min("ent_l2").alias("other"))
               .join(F.broadcast(cb), "other"))
    b = (d.where(F.col("dst_kind") == "CPTY").select(F.col("dst_id").alias("cpty_id"), F.lit("in").alias("dir"), "txn_date", "amt")
          .unionByName(d.where(F.col("src_kind") == "CPTY").select(F.col("src_id").alias("cpty_id"), F.lit("out").alias("dir"),
                                                                   "txn_date", "amt"))
          .join(mp, "cpty_id").drop("cpty_id").withColumn("scope", F.lit("ext")))
    return (a.unionByName(b).groupBy("scope", "other", "dir", "txn_date").agg(F.sum("amt").alias("amt"))
             .join(F.broadcast(CAL), "txn_date").withColumn("x", F.log1p("amt")).drop("amt", "txn_date"))

t0 = time.time()
SER, rebuilt_d = extract(f"{P_TMP}/evid_daily", SIG_D, build_daily)
st = (SER.groupBy("scope", "other", "dir").agg(F.count("*").alias("n"), F.sum("x").alias("S"),
                                               F.sum(F.col("x") * F.col("x")).alias("SS")).toPandas())
lag = F.explode(F.array(*[F.lit(L) for L in LAGS])).alias("L")
cross = []
for kind, lead, follow in [("fwd", "in", "out"), ("rev", "out", "in")]:
    a_ = SER.where(F.col("dir") == lead).select("scope", "other", "wi", F.col("x").alias("xa"), lag) \
            .withColumn("wi", F.col("wi") + F.col("L"))
    b_ = SER.where(F.col("dir") == follow).select("scope", "other", "wi", F.col("x").alias("xb"))
    cross.append(a_.join(b_, ["scope", "other", "wi"]).groupBy("scope", "other", "L")
                   .agg(F.sum(F.col("xa") * F.col("xb")).alias("Sxy")).withColumn("kind", F.lit(kind)).toPandas())
CR = pd.concat(cross)
stw = st.pivot_table(index=["scope", "other"], columns="dir", values=["n", "S", "SS"]).fillna(0)
stw.columns = [f"{a}_{b}" for a, b in stw.columns]
CR = CR.merge(stw.reset_index(), on=["scope", "other"])
lead_s, foll_s = np.where(CR.kind == "fwd", "in", "out"), np.where(CR.kind == "fwd", "out", "in")
Sx = np.where(CR.kind == "fwd", CR.S_in, CR.S_out); Sy = np.where(CR.kind == "fwd", CR.S_out, CR.S_in)
Sxx = np.where(CR.kind == "fwd", CR.SS_in, CR.SS_out); Syy = np.where(CR.kind == "fwd", CR.SS_out, CR.SS_in)
den = np.sqrt(np.clip(N_WD * Sxx - Sx ** 2, 0, None) * np.clip(N_WD * Syy - Sy ** 2, 0, None))
CR["r"] = np.where(den > 0, (N_WD * CR.Sxy - Sx * Sy) / np.where(den > 0, den, 1), np.nan)
best = CR.sort_values("r", ascending=False).groupby(["scope", "other", "kind"]).head(1)
EV_D = best.pivot_table(index=["scope", "other"], columns="kind", values=["r", "L"]).reset_index()
EV_D.columns = ["scope", "other"] + [f"{a}_{b}" for a, b in EV_D.columns[2:]]
EV_D = EV_D.merge(stw.reset_index()[["scope", "other", "n_in", "n_out"]], on=["scope", "other"], how="left")
for c in ["r_fwd", "r_rev", "L_fwd"]:
    if c not in EV_D:
        EV_D[c] = np.nan
EV_D["lead_ok_days"] = (EV_D.n_in >= LAG_MIN_DAYS) & (EV_D.n_out >= LAG_MIN_DAYS)
show(f"§10b Lead of money in over money out ({(time.time() - t0) / 60:.1f} min, {'built' if rebuilt_d else 'reused'} "
     f"extract): {N_WD} weekdays",
     ("Coverage", pd.DataFrame([
         {"item": "candidates with daily series", "value": f"{EV_D.other.nunique():,}"},
         {"item": f"with ≥ {LAG_MIN_DAYS} weekdays of money in and of money out", "value": f"{int(EV_D.lead_ok_days.sum()):,}"}])),
     note("r = correlation of log daily amounts on weekdays; fwd = money in today vs money out L days later, "
          "rev = the reverse. Shared month-end rhythm raises both, so the check needs fwd clearly above rev."))

In [ ]:
# §10c  Combine, test each check on the known examples, show the lists with evidence
EV = (pd.DataFrame(KEYS[KEYS.is_cand][["scope", "other"]])
        .merge(EV_M, on=["scope", "other"], how="left").merge(EV_D, on=["scope", "other"], how="left"))
EV["switch_n"] = EV.switch_n.fillna(0).astype(int)
EV["payroll_n"] = EV.payroll_n.fillna(0).astype(int)
EV["ev_switch"] = EV.switch_n >= SWITCH_MIN_CLIENTS
EV["ev_payroll"] = EV.payroll_n >= PAYROLL_MIN_CLIENTS
EV["ev_lead"] = (EV.lead_ok_days.fillna(False).astype(bool) & (EV.r_fwd >= LAG_MIN)
                 & (EV.r_fwd - EV.r_rev.fillna(-1) >= LAG_EDGE))
EV["ev_shared"] = (EV.shared_clients.fillna(0) >= OVERLAP_MIN_CLIENTS) & (EV.shared_lift.fillna(0) >= OVERLAP_LIFT)
CHECKS = ["ev_switch", "ev_payroll", "ev_lead", "ev_shared"]
EV["evidence_n"] = EV[CHECKS].sum(axis=1).astype(int)
EV["data_confirmed"] = EV.evidence_n >= EVIDENCE_MIN

def ev_text(r):
    t = []
    if r.ev_switch:
        t.append(f"{r.switch_n} clients switched from {r.switch_from}")
    if r.ev_payroll:
        t.append(f"{r.payroll_n} clients stopped paying staff directly")
    if r.ev_lead:
        t.append(f"money in leads out by {int(r.L_fwd)}d (r {r.r_fwd:.2f} vs {r.r_rev:.2f})")
    if r.ev_shared:
        t.append(f"clients use other processors {r.shared_lift:.1f}× as often")
    return "; ".join(t)
EV["evidence"] = EV.apply(ev_text, axis=1)
EVC = ["switch_n", "switch_from", "payroll_n", "r_fwd", "r_rev", "L_fwd", "shared_clients", "shared_lift"] + CHECKS + \
      ["evidence_n", "data_confirmed", "evidence"]
ev_a = EV[EV.scope == "int"].rename(columns={"other": "mdm_id"})[["mdm_id"] + EVC]
ev_b = EV[EV.scope == "ext"].rename(columns={"other": "ent_id"})[["ent_id"] + EVC]
CS = CS.drop(columns=[c for c in EVC if c in CS]).merge(ev_a, on="mdm_id", how="left")
P = P.drop(columns=[c for c in EVC if c in P]).merge(ev_b, on="ent_id", how="left")
for D_ in (CS, P):
    for c in CHECKS + ["data_confirmed"]:
        D_[c] = D_[c].fillna(False).astype(bool)
    D_["evidence_n"] = D_.evidence_n.fillna(0).astype(int)
    D_["evidence"] = D_.evidence.fillna("")
LIST_C = LIST_C.drop(columns=[c for c in EVC if c in LIST_C]).merge(ev_a, on="mdm_id", how="left")
LIST_P = LIST_P.drop(columns=[c for c in EVC if c in LIST_P]).merge(ev_b, on="ent_id", how="left")
for D_ in (LIST_C, LIST_P):
    D_["evidence"] = D_.evidence.fillna("")
    D_["evidence_n"] = D_.evidence_n.fillna(0).astype(int)

def check_table(D_, y, lab_pos, lab_neg):
    rows = []
    for c in CHECKS + ["data_confirmed"]:
        a, b = D_.loc[y, c].mean(), D_.loc[~y, c].mean()
        rows.append({"check": c.replace("ev_", ""), f"% {lab_pos} passing": 100 * a, f"% {lab_neg} passing": 100 * b,
                     "times as often": a / b if b > 0 else np.nan})
    return pd.DataFrame(rows)
VAL_A = check_table(CS, CS.is_tppp.to_numpy(bool), "registered TPPPs", "other screened clients")
VAL_B = check_table(P, P.example.to_numpy(bool), "known processors", "other screened companies")

def evidence_view_c(d):
    return pd.DataFrame({"rank": d["rank"], "customer": d.customer_name, "triage": d.triage,
                         "evidence": d.evidence_n, "what the data shows": d.evidence})
def evidence_view_p(d):
    return pd.DataFrame({"rank": d["rank"], "entity": d.ent_name, "relationship": d.relationship,
                         "known processor": d.known_proc, "evidence": d.evidence_n, "what the data shows": d.evidence})
low_rank = (P[~P.tppp_ext & ~P.cpty_excl & ~P.ent_id.isin(LIST_P.ent_id) & P.data_confirmed]
              .sort_values(["evidence_n", "score"], ascending=False).head(SHOW_N))
show("§10c Evidence from the data",
     ("Look-alikes: how often each check passes", VAL_A),
     ("Prospects: how often each check passes", VAL_B),
     note("A check is useful when examples pass it many times as often as everyone else; one that examples pass about "
          "as often as others is noise and should be ignored when reading the lists."),
     (f"Look-alikes top {SHOW_N} with evidence", evidence_view_c(LIST_C.head(SHOW_N))),
     (f"Prospects top {SHOW_N} with evidence", evidence_view_p(LIST_P.head(SHOW_N))),
     (f"Prospects outside the top {TOP_N_CPTY} that the data confirms ({len(low_rank)})",
      evidence_view_p(low_rank.assign(rank=low_rank.screened_rank))) if len(low_rank) else None)

## §11 Export for external enrichment

Files for looking companies up outside PNC (web research, business-information vendors). They hold a **random-looking
row id and the name only** — no power ids, amounts, scores, counterparties or relationship to PNC. The key that joins
row ids back stays inside PNC. **Get sign-off before any file leaves the bank**; the prospects file (outside companies)
is the lower-risk one, the look-alikes file names PNC clients. Generic settlement descriptors are left out (a web search
cannot tell who is behind them).

In [ ]:
# §11  De-identified export: row id + name (+ NAICS for look-alikes); internal key file
def rid(x):
    return hashlib.sha256(f"{VERSION}|{x}".encode()).hexdigest()[:12]
xb = LIST_P[~LIST_P.generic_name].head(ENRICH_TOP_N)
xb = pd.DataFrame({"row_id": xb.ent_id.map(rid), "list": "prospects", "name": xb.ent_name, "pkg_id": xb.ent_id})
xa = LIST_C.head(ENRICH_TOP_N)
xa = pd.DataFrame({"row_id": xa.mdm_id.map(rid), "list": "look-alikes", "name": xa.customer_name,
                   "naics_cd": xa.naics_cd, "pkg_id": xa.mdm_id})
f_xb = f"{OUT_DIR}/enrich_prospects_names.csv"
f_xa = f"{OUT_DIR}/enrich_lookalikes_names.csv"
f_xk = f"{OUT_DIR}/enrich_key_INTERNAL.csv"
xb.drop(columns=["pkg_id"]).to_csv(f_xb, index=False)
xa.drop(columns=["pkg_id"]).to_csv(f_xa, index=False)
pd.concat([xb[["row_id", "list", "pkg_id"]], xa[["row_id", "list", "pkg_id"]]]).to_csv(f_xk, index=False)
show("§11 Enrichment export",
     ("Files", pd.DataFrame([
         {"file": f_xb, "rows": len(xb), "may leave PNC": "after sign-off"},
         {"file": f_xa, "rows": len(xa), "may leave PNC": "after sign-off (names PNC clients)"},
         {"file": f_xk, "rows": len(xb) + len(xa), "may leave PNC": "never"}])),
     ("Prospects file, first rows", xb.drop(columns=["pkg_id"]).head(5)))

## §12 Outputs and summary

Writes the three tables and the review CSVs (with empty `verdict`, `note`, `reviewer` columns: every reviewed row
becomes a label for the next version). Access to the tables and files should stay with the TPPP / TM working group
until the permissible-use review is done.

| Result | Reading |
|---|---|
| a model beats the single best feature in §4 / §7 | the combination adds something; otherwise ship the one-feature rule, it is easier to explain |
| `shape` and `full` give similar recall in §4 | size is not doing the work; keep `shape` so the lists are not just large companies |
| many `behaviour only` names in §5 | unregistered processors or look-alikes (PEOs, payroll-heavy employers): the review decides, and the verdicts become labels |
| many `declared only` names in §5 | NAICS says processor but the money does not move like one: a KYC classification question, not a sales one |
| §8 top 50 mostly known processors | the ranking works; the value is in the unknown names between them and in the warm-introduction column |
| a check in §10c that examples pass ≥ 3× as often as others | trust it on the lists; `data_confirmed` names go to review first |
| §9b last-two-months ratio well above 1 | quote §9 at the January–June level |

In [ ]:
# §12  Write outputs, headline numbers
def tidy(d, keep):
    out = d[[c for c in keep if c in d.columns]].copy()
    for c in out.columns:
        if out[c].dtype == bool:
            out[c] = out[c].astype(bool)
    return out

base_c = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "naics_cd", "naics_desc", "months_active"]
flags_c = ["is_tppp", "tppp_family", "naics_proc", "name_proc", "pnc_own", "bank_like", "gov", "excluded",
           "sector_unlikely", "behaviour", "declared", "triage", "screened_rank"] + EVC
score_c = [c for c in CS.columns if c.startswith("pct_")] + ["score", "nearest_tppp", "nearest_tppp_dist", "reasons"]
OUT_C = tidy(CS, base_c + flags_c + score_c + [f for f in CUST_COL]).assign(window=f"{WINDOW_START}..{WINDOW_END}",
                                                                            version=VERSION)
base_p = ["ent_id", "ent_name", "name_class", "fi_dom", "months_active", "pnc_match_n", "pnc_tm_match_n",
          "n_ids_lifetime"]
flags_p = ["known_proc", "tppp_ext", "cpty_excl", "example", "pnc_name_match", "generic_name", "screened_rank",
           "relationship", "pnc_match_names"] + EVC
score_p = [c for c in P.columns if c.startswith("pct_")] + ["score", "nearest_example", "nearest_example_dist", "reasons"]
OUT_P = tidy(P, base_p + flags_p + score_p + CPTY_FEATS + ["ncust_all"]).assign(window=f"{WINDOW_START}..{WINDOW_END}",
                                                                                 version=VERSION)
OUT_I = INTRO.rename(columns={"customer_name": "tm_client", "customer_pwr_id": "tm_client_pwr_id"})[
    ["ent_id", "intro_rank", "mdm_id", "tm_client", "tm_client_pwr_id", "master_pwr_id", "naics_cd", "paid",
     "received", "total", "months"]].merge(LIST_P[["ent_id", "rank", "ent_name"]], on="ent_id")
written = []
if WRITE_TABLES:
    for pdf, name in [(OUT_C, "pkg_tppp_cust_scores"), (OUT_P, "pkg_tppp_cpty_scores"), (OUT_I, "pkg_tppp_cpty_intros")]:
        write_table(pdf, name)
        written.append({"table": f"{DB}.{name}", "rows": len(pdf)})
review = {"verdict": "", "note": "", "reviewer": "", "reviewed_at": ""}
f_c = f"{OUT_DIR}/tppp_lookalikes_top{TOP_N_CUST}.csv"
f_d = f"{OUT_DIR}/tppp_lookalikes_declared_only.csv"
f_p = f"{OUT_DIR}/tppp_prospects_top{TOP_N_CPTY}.csv"
f_9 = f"{OUT_DIR}/tppp_registered_business_elsewhere.csv"
cust_view(LIST_C).assign(mdm_id=LIST_C.mdm_id.values, **review).to_csv(f_c, index=False)
cust_view(decl).drop(columns=["rank"]).assign(mdm_id=decl.mdm_id.values, **review).to_csv(f_d, index=False)
cpty_view(LIST_P).assign(ent_id=LIST_P.ent_id.values, **review).to_csv(f_p, index=False)
if len(v9):
    v9.to_csv(f_9, index=False)
if len(V9B):
    V9B.to_csv(f"{OUT_DIR}/tppp_registered_business_elsewhere_monthly.csv", index=False)
VAL_A.assign(list="look-alikes").to_csv(f"{OUT_DIR}/evidence_checks.csv", index=False)
VAL_B.assign(list="prospects").to_csv(f"{OUT_DIR}/evidence_checks.csv", mode="a", header=False, index=False)
REC_C.to_csv(f"{OUT_DIR}/cust_recall.csv", index=False)
REC_P.to_csv(f"{OUT_DIR}/cpty_recall.csv", index=False)

def found(rec, k=0.01, group="all examples", method="blend"):
    r = rec[(rec.method == method) & (rec.examples == group)]
    col = next(c for c in rec.columns if c.startswith(f"found in top {100 * k:g}%"))
    return (int(r[col].iloc[0]), int(r.n.iloc[0])) if len(r) else (0, 0)

a1, an = found(REC_C, method=FINAL_C)
a5, _ = found(REC_C, 0.05, method=FINAL_C)
b1, bn = found(REC_P, group="known processors", method=FINAL_P)
b5, _ = found(REC_P, 0.05, group="known processors", method=FINAL_P)
head = pd.DataFrame([
    {"item": "registered TPPPs matched / screened", "value": f"{len(TPPP_IDS)} / {int(yC.sum())}"},
    {"item": "look-alikes screened TM organisations", "value": f"{len(CS):,}"},
    {"item": f"look-alikes held-out TPPPs found in the top 1% / 5% ({FINAL_C})", "value": f"{a1} / {a5} of {an} "
     f"({int(round(0.01 * len(CS))):,} / {int(round(0.05 * len(CS))):,} rows to review)"},
    {"item": "look-alikes candidates: behaviour + declared / behaviour only / declared only / behaviour, sector unlikely",
     "value": " / ".join(str(int((cand_all.triage == t).sum())) for t in
                         ["behaviour + declared", "behaviour only", "declared only", "behaviour, sector unlikely"])},
    {"item": "prospects screened external entities", "value": f"{len(P):,}"},
    {"item": f"prospects held-out known processors found in the top 1% / 5% ({FINAL_P})", "value": f"{b1} / {b5} of {bn}"},
    {"item": "prospects top-50 not on the known list", "value": f"{int((~LIST_P.head(50).known_proc).sum())}"},
    {"item": f"prospects top {len(LIST_P)}: new to bank / possible / existing PNC client",
     "value": " / ".join(str(int((LIST_P.relationship == r).sum())) for r in
                         ["new to bank", "possible PNC client", "existing PNC client"])},
    {"item": f"data-confirmed (≥ {EVIDENCE_MIN} checks): look-alikes top {len(LIST_C)} / prospects top {len(LIST_P)}",
     "value": f"{int((LIST_C.evidence_n >= EVIDENCE_MIN).sum())} / {int((LIST_P.evidence_n >= EVIDENCE_MIN).sum())}"},
    {"item": "registered TPPPs with same-name outside accounts", "value": f"{len(W9)}"},
])
files = pd.DataFrame({"file": [f_c, f_d, f_p, f_9 if len(v9) else "(none)", f_xb, f_xa, f_xk, f"{OUT_DIR}/cust_recall.png",
                               f"{OUT_DIR}/cpty_recall.png", f"{OUT_DIR}/cust_feature_auc.png"]})
show("§12 Summary",
     ("Headline", head),
     ("Tables written", pd.DataFrame(written)) if written else note("WRITE_TABLES = False: no tables written."),
     ("Files", files),
     note("Impala: " + " ".join(f"INVALIDATE METADATA {w['table']};" for w in written)) if written else None,
     note("Look-alikes top 5: " + "; ".join(LIST_C.customer_name.head(5).fillna("?"))),
     note("Prospects top 5 new to bank, not on the known list: " + "; ".join(
         LIST_P[~LIST_P.known_proc & (LIST_P.relationship == "new to bank")].ent_name.head(5).fillna("?"))))